# [이론] 피처 선택 · 데이터 누수

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🔧 피처·파이프라인·데이터 누수 — 새지 않는 실험 설계

## — 좋은 입력을 만들고 평가 정보가 새지 않게 묶습니다

---

## 📋 학습 목표

이 학습 노트북을 마치면 여러분은 다음을 할 수 있습니다.

1. 파생변수를 만들고 범주형을 **인코딩**하고 수치형을 **스케일링**하는 작업을 scikit-learn 문법으로 수행할 수 있습니다.
2. **데이터 누수(Data Leakage)** 의 대표 유형을 알고, 점수가 부풀려지는 장면을 직접 재현할 수 있습니다.
3. `Pipeline`과 `ColumnTransformer`로 전처리와 모델을 하나로 묶어 **전처리 누수의 위험을 구조적으로 낮출** 수 있습니다.
4. `Pipeline` 전체를 `GridSearchCV`에 넣어 **전처리 기준이 검증 조각에 섞이지 않게** 튜닝하고, 결과를 **실험 기록표**로 남길 수 있습니다.
5. 학습된 `Pipeline`을 통째로 저장·재로드해 예측이 재현되는지 확인하고, 피처·모델·튜닝의 효과를 같은 검증 조건에서 비교할 수 있습니다.

_💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해 봅시다. 노트북을 마친 뒤 다시 돌아와 비교하면 성장이 눈에 보일 것입니다._

## 📚 학습 목차

| Part | 내용                                     | 핵심 질문                                  | 예상 시간 |
| ---- | ---------------------------------------- | ------------------------------------------ | --------- |
| 0    | 오늘의 여정                              | 지표는 골랐다. 그런데 입력이 부실하다면?   | 15분      |
| 1    | 피처를 만든다                            | 없던 변수를 만들고 글자를 숫자로 바꾸는 법 | 45분      |
| 2    | 데이터 누수                              | 점수는 좋은데 실전에서 무너지는 이유는?    | 35분      |
| 3    | `Pipeline` — 전처리 누수를 예방한다      | 전처리의 `fit` 범위를 어떻게 고정하는가?   | 55분      |
| 4    | 평가 정보를 분리한 실험 — 튜닝·기록·저장 | 어떤 레버를 먼저 당겨야 하는가?            | 60분      |
| 정리 | 핵심 요약 · 실습 안내                    | 실험을 어떻게 남길 것인가                  | 20분      |

_⏱️ 예상 시간은 참고용입니다. 학습 속도는 사람마다 다릅니다 — 여러분의 속도로 가면 됩니다._

# Part 0. 오늘의 여정

## 🔁 지난 시간 복습

지난 두 번에 걸쳐 여러분은 **재는 눈**을 갖췄습니다. 교차 검증으로 점수의 흔들림까지 함께 보고하는 법을 배웠고, 불균형 데이터에서 정확도만 사용할 때의 한계를 확인하고 목적에 맞는 지표를 선택하는 법을 익혔습니다.

이제 여러분의 성능 보고는 정직합니다. 그런데 한 가지가 남았습니다 — **재는 대상 자체**입니다.

아무리 정확한 자로 재도, 재는 물건이 부실하면 의미가 없습니다. 지금까지 우리는 데이터가 이미 잘 준비된 상태에서 시작했습니다. 오늘은 그 앞 단계로 갑니다.

## 🗺️ 오늘 배울 것

오늘은 두 개의 이야기가 하나로 합쳐집니다.

**첫째, 입력을 만드는 일.** 없던 변수를 만들고(파생변수), 글자를 숫자로 바꾸고(인코딩), 단위를 맞춥니다(스케일링). 사실 이 작업은 여러분이 이미 pandas로 해본 것들입니다. 오늘은 같은 일을 **scikit-learn 문법으로 옮깁니다.** 새 개념이 아니라 새 문법입니다.

**둘째, 그 일을 하다가 저지르는 사고.** 전처리는 위험한 작업입니다. 순서를 한 번 잘못 놓으면 테스트 데이터의 정보가 학습에 슬쩍 새어 들어가고, 그러면 **평가 점수가 실제 일반화 성능보다 낙관적으로 나타날 수 있습니다.** 이것을 **데이터 누수**라고 합니다.

오늘은 전처리와 모델을 `Pipeline`으로 묶어, 교차 검증의 각 학습 조각에서만 전처리 기준을 학습하도록 구성합니다.

> 🧭 **한눈에 보기**  
> 오늘의 줄기는 **"만들고 → 새는지 확인하고 → 새지 않는 구조로 묶는다"** 입니다. Part 1이 만들기, Part 2가 누수 진단, Part 3이 `Pipeline`으로 봉인, Part 4가 그 봉인된 파이프 안에서 실험하고 기록하는 법입니다. 마지막에는 이 데이터와 검증 조건에서 피처·모델·튜닝의 변화량을 비교하고 다음 실험의 우선순위를 정합니다.

## 🎬 오늘의 무대 — 개발 점수와 운영 성능이 달라진 가상 사례

여러분의 팀에 이런 일이 있었습니다.

한 팀이 이탈 예측 모델의 교차 검증 정확도 **0.89**를 확인하고 운영에 적용했다고 가정합니다. 한 달 뒤 같은 지표가 **0.71**로 낮아졌습니다. 이 수치는 누수의 원리를 설명하기 위한 가상 사례입니다.

코드를 뒤져보니 딱 한 줄이 문제였습니다. 전처리를 **전체 데이터에 먼저 적용**하고 그다음 교차 검증을 돌린 것입니다. 순서가 바뀌면서 검증 데이터의 분포 정보가 학습 과정에 들어갔고, 개발 점수는 실제 운영 성능을 제대로 추정하지 못했습니다.

이런 사고에는 이름이 있습니다. **데이터 누수**입니다. 누수는 코드가 정상 실행되고 점수도 좋아 보일 수 있어 검토 과정에서 놓치기 쉽습니다.

오늘은 누수가 평가에 미치는 영향을 재현하고, 전처리의 `fit` 범위를 관리하는 구조를 익힙니다.

## 🤖 오늘의 AI 활용 규칙 — 검증 단계

AI에게 코드 생성을 요청할 수 있습니다. 다만 오늘은 특히 조심해야 합니다.

- **허용:** 전처리 코드, `Pipeline` 구성, `GridSearchCV` 설정을 AI에게 생성 요청
- **의무:** 채택 전에 **누수 검사**를 통과시킵니다. 그리고 모델 카드에 'AI 사용 내역' 세 줄을 기록합니다.
- **오늘 AI가 가장 자주 만드는 사고:** AI가 `scaler.fit_transform(X)`를 **전체 데이터에** 적용한 다음 `train_test_split`을 수행하는 코드를 제안할 수 있습니다. 이것이 정확히 오늘의 무대에서 벌어진 그 사고입니다.

> ⚠️ **주의하기**  
> AI가 제안한 전처리 코드는 **`fit`의 대상, 피처의 생성 시점, 분할 단위**를 확인합니다. 평가에 사용할 조각까지 포함해 전처리 기준을 학습했다면 누수입니다.

> 💡 **핵심짚기**  
> 이 코드에 대해 ① **왜** 이렇게 짰는지 ② **어디서** 틀릴 수 있는지 ③ 결과를 **왜 믿는지** — 세 질문에 답할 수 있으면 여러분 것입니다. 답할 수 없다면 채택하기 전에 근거와 검증 절차를 보완해야 합니다.

## ⚙️ 환경 설정과 실습 데이터

> ⚠️ **주의하기**  
> 코드 셀은 반드시 위에서부터 순서대로 실행합니다. 중간부터 실행하면 앞에서 만든 변수가 없어 오류가 납니다.

▶️ **코드 실행하기 · 코드 셀 1 [C1]**

In [1]:
# ─────────────────────────────────────────────
# [C1] Part 0 · ⚙️ 환경 설정과 실습 데이터
# 환경 준비 — 라이브러리 + 한글 폰트 + 시드 고정 (실행하면 버전과 폰트가 출력됩니다)
# ─────────────────────────────────────────────
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

warnings.filterwarnings("ignore")

# 오늘 쓰는 도구 — 전처리·조립·검증
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, GridSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   OneHotEncoder, OrdinalEncoder, TargetEncoder)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib

# 실습 데이터는 인터넷에서 바로 읽어옵니다 (코랩·로컬 어디서든 같은 코드로 동작)
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 한글 폰트 설정 — 그래프의 한글이 □□□로 깨지지 않게 합니다
system = platform.system()
if system == "Darwin":                      # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                                        # 코랩 등 리눅스 — 나눔고딕을 설치해야 합니다
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        print("나눔고딕을 설치합니다 (10~20초 걸립니다)...")
        import subprocess
        subprocess.run("apt-get install -y -qq fonts-nanum", shell=True,
                       capture_output=True)
        for path in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
            fm.fontManager.addfont(path)
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font=plt.rcParams["font.family"])

print("환경 준비 완료!")
import sklearn
print(f"- scikit-learn {sklearn.__version__}")
print(f"- 그래프 한글 폰트: {plt.rcParams['font.family'][0]}")

환경 준비 완료!
- scikit-learn 1.9.0
- 그래프 한글 폰트: Malgun Gothic


### 데이터 준비 — Titanic 승객 데이터

Concept에서는 `seaborn-data` 저장소의 **Titanic 승객 데이터**를 사용합니다. 한 행은 승객 1명이며, 탑승 시점에 알 수 있는 정보를 이용해 생존 여부를 분류하는 학습 예제입니다.

| 항목           | 내용                                                                            |
| -------------- | ------------------------------------------------------------------------------- |
| 관측 단위      | 승객 1명                                                                        |
| 원본 크기      | 891행 × 15열                                                                    |
| 이번 실습 범위 | 891행 × 9열 — 입력 피처 8개 + 타깃 1개                                          |
| 타깃           | `survived` — 생존 1, 사망 0 · 양성 비율 38.4%                                   |
| 수치형 입력    | `pclass`, `age`, `sibsp`, `parch`, `fare` — 5개                                 |
| 범주형 입력    | `sex`, `embarked`, `who` — 3개                                                  |
| 결측치         | `age` 177건, `embarked` 2건                                                     |
| 사용 목적      | 혼합형 전처리, 결측 대치, 파생변수, `ColumnTransformer`와 `Pipeline`, 누수 비교 |

> ℹ️ **출처 확인하기**  
> 실습 파일은 [`seaborn-data/titanic.csv`](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv)에서 불러옵니다. 이 저장소는 Seaborn 예제용 사본이며 일부 데이터가 원출처와 다를 수 있습니다. 저장소가 밝힌 Titanic의 원출처는 [Kaggle Titanic 데이터](https://www.kaggle.com/c/titanic/data)입니다.

`who`는 승객 유형을 정리한 편의 열입니다. 이 노트북의 목적은 역사적 생존 예측 모델을 운영하는 것이 아니라, **서로 다른 자료형과 결측치를 같은 파이프라인에서 안전하게 처리하는 과정**을 익히는 데 있습니다.

▶️ **코드 실행하기 · 코드 셀 2 [C2]**

In [2]:
# ─────────────────────────────────────────────
# [C2] Part 0 · ⚙️ 환경 설정과 실습 데이터
# 타이타닉 로드 — 수치+범주 혼합, 결측 있음 (실행하면 앞 5행과 결측 현황이 나옵니다)
# ─────────────────────────────────────────────
titanic = pd.read_csv(DATA_URL + "titanic.csv")

# 오늘 쓸 컬럼만 골라냅니다
df = titanic[["survived", "pclass", "sex", "age", "sibsp", "parch",
              "fare", "embarked", "who"]].copy()

print(f"전체 {df.shape[0]}명 × 컬럼 {df.shape[1]}개")
print(f"생존율 {df['survived'].mean():.4f}   (타깃: survived)")
print()
print(df.head(5).to_string())
print()
print("타입별 컬럼")
print(f"  수치형: {df.select_dtypes(include='number').columns.tolist()}")
print(f"  범주형: {df.select_dtypes(exclude='number').columns.tolist()}")
print()
na = df.isna().sum()
print("결측 있는 컬럼")
print(na[na > 0].to_string())
print()
print("→ age에 결측이 177건 있습니다. 이것을 어떻게 채울지도 오늘의 주제입니다.")

전체 891명 × 컬럼 9개
생존율 0.3838   (타깃: survived)

   survived  pclass     sex   age  sibsp  parch     fare embarked    who
0         0       3    male  22.0      1      0   7.2500        S    man
1         1       1  female  38.0      1      0  71.2833        C  woman
2         1       3  female  26.0      0      0   7.9250        S  woman
3         1       1  female  35.0      1      0  53.1000        S  woman
4         0       3    male  35.0      0      0   8.0500        S    man

타입별 컬럼
  수치형: ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
  범주형: ['sex', 'embarked', 'who']

결측 있는 컬럼
age         177
embarked      2

→ age에 결측이 177건 있습니다. 이것을 어떻게 채울지도 오늘의 주제입니다.


> 👀 **[C2] 결과읽기**  
> 수치형 5개(`pclass`·`age`·`sibsp`·`parch`·`fare`)와 범주형 3개(`sex`·`embarked`·`who`)가 섞여 있습니다. `age`에 결측이 **177건**(약 20%), `embarked`에 2건 있습니다.

여기서 `who`는 승객 유형(`man`/`woman`/`child`)입니다. 나중에 이 컬럼이 흥미로운 역할을 합니다.

이제 이 원재료로 무엇을 할 수 있는지 봅니다.

# Part 1. 피처를 만든다 — 파생·인코딩·스케일링

머신러닝 모델에 데이터를 입력하려면 세 가지 조건을 점검합니다.

1. **전부 숫자여야 합니다.** 대부분의 scikit-learn 분류기는 `sex`의 `"male"`/`"female"` 문자열을 그대로 처리하지 못합니다.
2. **빈칸이 없어야 합니다.** `age`의 결측 177건을 채워야 합니다.
3. (모델에 따라) **단위가 비슷해야 합니다.** `fare`(0~512)와 `pclass`(1~3)가 섞여 있으면 거리 기반 모델이 흔들립니다.

그리고 여기에 하나를 더 얹을 수 있습니다 — **없던 변수를 만들어내는 것**입니다. 파생변수의 효과는 모델과 데이터에 따라 달라지므로 교차 검증으로 확인합니다.

> ❓ **질문하기**  
> 원재료를 모델이 먹을 수 있는 형태로 바꾸면서, 동시에 더 좋은 재료를 만들어내는 방법은?

## 💡 쉽게 말하면 — 재료를 손질해서 내놓기

같은 재료로 요리를 해도 손질에 따라 결과가 달라집니다. 통째로 넣는 것과 썰어서 넣는 것, 데쳐서 넣는 것이 다릅니다.

**파생변수**는 기존 열을 조합하거나 변환해 모델이 사용할 표현을 만드는 일입니다. `sibsp`(형제·배우자 수)와 `parch`(부모·자녀 수)를 따로 주면 모델은 각각의 영향만 봅니다. 그런데 이 둘을 더해 **`family_size`(가족 크기)** 를 만들어주면 "혼자 탔는가, 대가족인가"라는 관계를 한 열로 표현할 수 있습니다.

**인코딩**은 글자를 숫자로 바꾸는 일입니다. 여기에 함정이 있습니다 — 잘못 바꾸면 **없던 순서가 생깁니다.**

**스케일링**은 단위를 맞추는 일입니다. `fare`가 0~512이고 `pclass`가 1~3이면, 거리를 재는 모델에게는 `fare`가 100배 중요한 것처럼 보입니다.

> 🔗 **개념 연결하기**  
> 이 세 작업은 여러분이 **이미 pandas로 해본 것들**입니다. `df["a"] + df["b"]`, `pd.get_dummies()`, `(x - x.mean()) / x.std()` — 전부 익숙한 조작입니다. 오늘 배우는 것은 새 개념이 아니라 **같은 일을 sklearn 문법으로 옮기는 것**이고, 옮기는 이유는 Part 3에서 밝혀집니다.

## 🔍 자세히 알아보기 — 세 작업의 문법과 함정

### ① 파생변수 — 도메인 지식을 숫자로

좋은 파생변수의 조건은 세 가지입니다.

- **업무 가설이 분명해야** 합니다. 어떤 관계를 모델에 드러내려는지 설명할 수 있어야 합니다.
- **예측 시점에 사용할 수 있어야** 합니다. 결과가 발생한 뒤 생기는 정보라면 누수입니다.
- **추가 효과를 검증해야** 합니다. 원본 열의 재표현도 모델에 따라 유용할 수 있지만, 중복만 늘릴 수도 있습니다.

타이타닉에서 만들 수 있는 것들입니다.

| 파생변수          | 계산                 | 담은 가설                |
| ----------------- | -------------------- | ------------------------ |
| `family_size`     | `sibsp + parch + 1`  | 가족 규모가 생존에 영향  |
| `is_alone`        | `family_size == 1`   | 혼자 탄 사람은 다르다    |
| `fare_per_person` | `fare / family_size` | 1인당 지불액이 실제 지위 |

### ② 인코딩 — 없던 순서를 만들지 않기

범주형을 숫자로 바꾸는 두 가지 방법입니다.

| 방법        | 하는 일                | 문제                             |
| ----------- | ---------------------- | -------------------------------- |
| **Ordinal** | `S→0, C→1, Q→2`        | **근거 없는 순서와 간격**이 생김 |
| **One-Hot** | 범주마다 0/1 열을 따로 | 열이 늘어남 (범주 많으면 부담)   |

**순서가 실제로 있는 범주**(예: `저·중·고`, `초급·중급·고급`)에는 Ordinal이 맞습니다. **순서가 없는 범주**(승선 항구, 색깔, 지역)에는 One-Hot이 맞습니다.

`OneHotEncoder(handle_unknown="ignore")`를 설정합니다. 교차 검증 중 어떤 조각의 학습 부분에 특정 범주가 없을 수 있고, 그때 이 옵션이 없으면 오류로 멈춥니다.

### ③ 스케일링 — 단위를 맞춘다

| 스케일러         | 방식               | 쓰기 좋은 때                    |
| ---------------- | ------------------ | ------------------------------- |
| `StandardScaler` | 평균 0, 표준편차 1 | 선형·거리 기반 모델의 기본 후보 |
| `MinMaxScaler`   | 0~1 범위로         | 값의 범위를 고정해야 할 때      |
| `RobustScaler`   | 중앙값·사분위 사용 | **이상치가 많을 때**            |

**스케일링이 필요한 모델과 필요 없는 모델이 갈립니다.**

- **영향이 큼** — 거리 기반 모델(KNN)과 규제를 사용하는 선형 모델(로지스틱 회귀, SVM), 신경망
- **예측 결과에 대체로 불필요** — 트리 계열(의사결정나무, 랜덤포레스트, 부스팅). 단조 스케일 변환은 분할 순서를 바꾸지 않습니다

> ⚠️ **주의하기**  
> 결측 대치와 스케일링은 모두 **데이터로부터 기준값을 학습**하는 작업입니다. 중앙값이 얼마인지, 평균과 표준편차가 얼마인지를 계산해야 하니까요. **그 계산을 어느 데이터에서 하느냐**가 오늘 Part 2의 주제입니다. 미리 말하면 — 학습 조각에서만 해야 합니다.

> 📌 **실무 연결하기**  
> 파생변수는 산업 지식이 가장 직접적으로 값을 만드는 자리입니다 — **금융**의 부채비율·소득대비상환액, **이커머스**의 최근성·구매빈도·구매금액(RFM), **제조**의 가동시간당 불량률, **헬스케어**의 체질량지수(BMI)가 모두 원본 컬럼의 조합입니다. 도메인 전문가와의 대화가 모델 성능으로 직결되는 지점입니다.

## 📊 데이터로 확인해 봅시다 — 만들고, 바꾸고, 맞춰 보기

먼저 파생변수를 만들어 **가설이 맞는지** 확인합니다. `family_size`가 생존과 관계가 있을까요?

> 🤔 **예상하기**  
> 가족 크기와 생존율의 관계를 예상합니다. 혼자 탄 집단과 가족이 있는 집단의 차이, 가족 크기가 커질 때의 변화를 확인합니다.

▶️ **코드 실행하기 · 코드 셀 3 [C3]**

In [3]:
# ─────────────────────────────────────────────
# [C3] Part 1 · 📊 데이터로 확인해 봅시다 — 만들고, 바꾸고, 맞춰 보기
# 파생변수 3개 생성 → 가설 검증 (실행하면 가족 크기별 생존율이 나옵니다)
# ─────────────────────────────────────────────
# pandas로 만들던 그 조작입니다
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)
df["fare_per_person"] = (df["fare"] / df["family_size"]).round(2)

print("가족 크기별 생존율")
fam = df.groupby("family_size")["survived"].agg(["mean", "size"])
fam.columns = ["생존율", "인원"]
print(fam.round(3).to_string())
print()
print(f"혼자 탄 사람({int(df['is_alone'].sum())}명) 생존율 : "
      f"{df.loc[df['is_alone'] == 1, 'survived'].mean():.3f}")
print(f"가족과 탄 사람({int((1 - df['is_alone']).sum())}명) 생존율: "
      f"{df.loc[df['is_alone'] == 0, 'survived'].mean():.3f}")

가족 크기별 생존율
               생존율   인원
family_size            
1            0.304  537
2            0.553  161
3            0.578  102
4            0.724   29
5            0.200   15
6            0.136   22
7            0.333   12
8            0.000    6
11           0.000    7

혼자 탄 사람(537명) 생존율 : 0.304
가족과 탄 사람(354명) 생존율: 0.506


> 🎯 **[C3] 확인하기**  
> 관계가 **뒤집히는 산 모양**입니다.

| 가족 크기 | 생존율    | 인원 |
| --------- | --------- | ---- |
| 1 (혼자)  | **0.304** | 537  |
| 2         | 0.553     | 161  |
| 3         | 0.578     | 102  |
| 4         | **0.724** | 29   |
| 5         | 0.200     | 15   |
| 6         | 0.136     | 22   |
| 7         | 0.333     | 12   |
| 8         | **0.000** | 6    |
| 11        | **0.000** | 7    |

혼자 탄 사람의 생존율이 0.304로 낮고, 4인 가족이 **0.724**로 가장 높습니다. 그런데 5인 이상부터 급격히 떨어져 8인·11인 가족은 **전원 사망**입니다.

이 결과는 `sibsp`와 `parch`를 합친 가족 규모와 생존율 사이에 비선형 관계가 있을 수 있음을 보여줍니다. `family_size`가 실제 예측 성능을 높이는지는 원본 열만 사용한 모델과 같은 검증 조건에서 비교해야 합니다.

> ⚠️ **주의하기**  
> 5인 이상 구간의 인원이 각각 6~22명뿐입니다. **표본이 적은 구간의 비율은 크게 흔들립니다** — 8인 가족 6명의 결과를 모든 대가족으로 일반화할 수 없습니다. 비율과 함께 분모를 확인합니다.

이제 인코딩입니다. 교과서는 "명목형에 Ordinal을 쓰면 가짜 순서가 생겨 선형 모델이 망가진다"고 말합니다. 정말 그럴까요?

▶️ **코드 실행하기 · 코드 셀 4 [C4]**

In [4]:
# ─────────────────────────────────────────────
# [C4] Part 1 · 📊 데이터로 확인해 봅시다 — 만들고, 바꾸고, 맞춰 보기
# One-Hot vs Ordinal — embarked를 두 방식으로 인코딩해 비교 (실행하면 CV 점수가 나옵니다)
# ─────────────────────────────────────────────
y = df["survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
base_num = ["pclass", "age", "fare"]      # 효과를 보려고 일부러 최소 구성으로

encoders = [
    ("One-Hot", OneHotEncoder(handle_unknown="ignore")),
    ("Ordinal(Label)", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
]

for label, enc in encoders:
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), base_num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", enc)]), ["embarked"]),
    ])
    pipe = Pipeline([("pre", pre),
                     ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])
    s = cross_val_score(pipe, df[base_num + ["embarked"]], y, cv=cv, scoring="accuracy")
    print(f"  {label:16} CV 정확도 {s.mean():.4f} ± {s.std():.4f}")

print()
print(f"embarked의 범주: {sorted(df['embarked'].dropna().unique().tolist())} ({df['embarked'].nunique()}개)")

  One-Hot          CV 정확도 0.7003 ± 0.0231
  Ordinal(Label)   CV 정확도 0.7104 ± 0.0333

embarked의 범주: ['C', 'Q', 'S'] (3개)


> 👀 **[C4] 결과읽기**  
> 결과가 교과서와 다릅니다. 정직하게 짚고 갑니다.

| 인코딩  | CV 정확도           |
| ------- | ------------------- |
| One-Hot | 0.7003 ± 0.0231     |
| Ordinal | **0.7104** ± 0.0333 |

**이 실행에서는 Ordinal의 평균이 조금 높습니다.** "명목형에 Ordinal을 쓰면 성능이 떨어진다"는 예상이 이 데이터에서는 성립하지 않았습니다.

범주가 3개인 이 작은 비교에서는 `S`/`C`/`Q`를 0/1/2로 놓아 근거 없는 순서가 생겼지만, 두 평균의 차이는 0.010이며 겹별 변동도 큽니다. 이 한 번의 5겹 교차 검증만으로 어느 인코딩이 일반적으로 우월하다고 결론 내릴 수 없습니다.

그러면 One-Hot을 쓰라는 원칙은 폐기해야 할까요? **아닙니다.** 원칙이 지키는 것은 이 데이터의 0.01이 아니라 두 가지입니다.

1. **범주가 많아질수록 해악이 커집니다.** 범주가 20개인 지역 코드를 0~19로 놓으면, 선형 모델은 범주 번호가 한 단계 증가할 때마다 같은 방향과 크기로 효과가 변한다는 근거 없는 관계를 가정합니다.
2. **계수 해석이 무의미해집니다.** Ordinal로 넣은 계수는 "범주 번호가 1 올라갈 때"의 효과인데, 그 번호에 의미가 없으므로 해석할 수 없습니다.

> 💡 **핵심짚기**  
> 이 결과는 "원칙이 틀렸다"가 아니라 **"이 데이터에서는 원칙의 효과가 측정되지 않았다"** 는 뜻입니다. 명목형에 One-Hot을 사용하는 원칙은 불필요한 순서 가정을 피하기 위한 것입니다. 데이터에 따라 평균 점수 차이가 작거나 반대 방향으로 나타날 수도 있습니다. 따라서 점수 하나와 별개로 인코딩이 모델에 부여하는 가정을 확인해야 합니다.

이제 스케일링입니다. 트리에는 필요 없다고 했으니, **거리를 재는 모델(KNN)** 로 확인합니다.

▶️ **코드 실행하기 · 코드 셀 5 [C5]**

In [5]:
# ─────────────────────────────────────────────
# [C5] Part 1 · 📊 데이터로 확인해 봅시다 — 만들고, 바꾸고, 맞춰 보기
# 스케일링이 거리 기반 모델(KNN)에 미치는 영향 (실행하면 스케일러별 점수가 나옵니다)
# ─────────────────────────────────────────────
NUM_COLS = ["pclass", "age", "sibsp", "parch", "fare",
            "family_size", "is_alone", "fare_per_person"]

print("컬럼별 값의 범위 (스케일 차이 확인)")
rng = df[NUM_COLS].agg(["min", "max"]).T
rng["범위"] = rng["max"] - rng["min"]
print(rng.round(1).to_string())
print()

print("KNN(n_neighbors=5) 성능 — 스케일러별")
scalers = [(None, "스케일링 없음"), (StandardScaler(), "StandardScaler"),
           (MinMaxScaler(), "MinMaxScaler"), (RobustScaler(), "RobustScaler")]
for sc, label in scalers:
    steps = [("impute", SimpleImputer(strategy="median"))]
    if sc is not None:
        steps.append(("scale", sc))
    steps.append(("clf", KNeighborsClassifier(n_neighbors=5)))
    s = cross_val_score(Pipeline(steps), df[NUM_COLS], y, cv=cv, scoring="accuracy")
    print(f"  {label:16} {s.mean():.4f} ± {s.std():.4f}")

컬럼별 값의 범위 (스케일 차이 확인)
                 min    max     범위
pclass           1.0    3.0    2.0
age              0.4   80.0   79.6
sibsp            0.0    8.0    8.0
parch            0.0    6.0    6.0
fare             0.0  512.3  512.3
family_size      1.0   11.0   10.0
is_alone         0.0    1.0    1.0
fare_per_person  0.0  512.3  512.3

KNN(n_neighbors=5) 성능 — 스케일러별
  스케일링 없음          0.6958 ± 0.0180
  StandardScaler   0.7059 ± 0.0335
  MinMaxScaler     0.7014 ± 0.0336
  RobustScaler     0.7104 ± 0.0226


> 👀 **[C5] 결과읽기**  
> `fare`의 범위가 **512.3**인데 `is_alone`은 **1**입니다. 500배 차이입니다. 거리를 계산하면 `fare` 하나가 거리를 지배합니다.

| 스케일러         | KNN 정확도          |
| ---------------- | ------------------- |
| 스케일링 없음    | **0.6880** ± 0.0183 |
| `StandardScaler` | **0.7059** ± 0.0333 |
| `MinMaxScaler`   | 0.6969 ± 0.0381     |
| `RobustScaler`   | 0.7059 ± 0.0215     |

스케일링만으로 **+0.018** 올랐습니다. 모델도 피처도 그대로이고, 단위만 맞춘 결과입니다. `StandardScaler`와 `RobustScaler`가 같은 값(0.7059)인데 `RobustScaler`의 겹별 표준편차가 더 작게 관찰됩니다(±0.022 대 ±0.033). 이 한 번의 분할 결과만으로 이상치가 원인이라고 단정하지 않고, 반복 검증이나 분포 점검으로 확인합니다.

> 📌 **실무 연결하기**  
> 선형·거리 기반 모델에서는 `StandardScaler`를 첫 후보로 두고, 이상치와 분포를 확인한 뒤 `RobustScaler` 등을 비교할 수 있습니다. 이 실행에서는 스케일링 유무의 평균 차이가 스케일러 사이의 평균 차이보다 크게 나타났습니다.

## ⌨️ 백문이 불여일타! (1)

```
[문제]
파생변수를 하나 더 만들어 가설을 검증합니다.

1) age를 구간으로 나눈 age_group을 만듭니다.
   구간: 0~12(child) / 13~19(teen) / 20~59(adult) / 60~(senior)
   힌트: pd.cut()을 쓰면 한 줄입니다. 결측은 그대로 둡니다.
2) age_group별 생존율과 인원을 출력합니다.
3) 이 파생변수가 '좋은 피처의 조건' 세 가지를 만족하는지 판단합니다.
```

▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [6]:
# [C6] Part 1 · ⌨️ 백문이 불여일타! (1)
# ⌨️ 백문이 불여일타! (1) — age를 구간화해 새 피처 만들고 가설 검증

bins = [0, 12, 19, 59, 200]
labels = ["child", "teen", "adult", "senior"]

# 1) age_group 생성
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

# 2) age_group별 생존율과 인원 출력
print("age_group별 생존율")
age_grp = df.groupby("age_group")["survived"].agg(["mean", "size"])
age_grp.columns = ["생존율", "인원"]
print(age_grp.round(3).to_string())
print()

# 결측치(age가 NaN인 행) 개수도 확인
print(f"age_group이 결측인 인원(원래 age 결측): {df['age_group'].isna().sum()}명")

age_group별 생존율
             생존율   인원
age_group            
child      0.580   69
teen       0.411   95
adult      0.389  524
senior     0.269   26

age_group이 결측인 인원(원래 age 결측): 177명


- ① 업무 가설이 분명한가? 넵
- ② 예측 시점에 사용할 수 있는가? 넵 탑승 시점에 이미 알 수 있는 정보임
- ③ 추가 효과가 검증되는가? age와 정보가 상당 부분 중복된다. 실제로 유용한지는 age_group을 추가했을 때 CV 점수가 age만 사용했을 때보다 오르는지 cv_of 같은 함수로 같은 검증 조건에서 비교해봐야한다.

<details>
<summary>(클릭) 💡 힌트</summary>

- `pd.cut(df["age"], bins=bins, labels=labels)`로 구간을 만들 수 있습니다.
- 생존율과 인원을 함께 보려면 `df.groupby("age_group")["survived"].agg(["mean", "size"])`를 사용합니다.
- 3번의 세 조건은 §🔍에 있습니다 — 업무 가설이 분명한가 / 예측 시점에 사용 가능한가 / 추가 효과가 검증되는가.

</details>

## 🚦 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어갑니다. 틀려도 괜찮습니다.

1. 파생변수를 검토하는 세 조건을 말할 수 있습니까? 가설분명한지/예츠 시점에 사용할수잇는지/추가 효과가 검증되는지
2. 명목형 범주에 Ordinal 인코딩을 쓰면 안 되는 이유를, 오늘 결과가 미미했던 것과 함께 설명할 수 있습니까? /의도하지 않은 순서가 생김. 범주가 커질수록 왜곡이 커지고, 계수 해석도 무의미해짐
3. 스케일링이 필요한 모델과 필요 없는 모델을 각각 예로 들 수 있습니까? KNN 의사결정나무

> ⏭️ **다음 학습 예고**  
> 재료를 손질하는 법을 배웠습니다. 그런데 지금 우리가 한 작업에는 위험한 것이 하나 섞여 있습니다 — **결측 대치와 스케일링은 데이터로부터 기준값을 학습합니다.** 그 학습을 어느 데이터에서 하는지가 점수의 진실성을 결정합니다. 다음 Part에서 오늘의 무대에서 벌어진 그 사고를 직접 재현합니다.

# Part 2. 데이터 누수 — 평가를 왜곡하는 사고

Part 0의 가상 사례에서는 교차 검증 점수와 운영 성능이 크게 달랐습니다.

이런 격차의 원인 가운데 하나는 **평가용 정보가 모델 개발 과정에 들어가는 데이터 누수**입니다. 분포 이동이나 데이터 품질 변화 같은 다른 원인도 별도로 점검해야 합니다. 시험 문제를 미리 본 학생이 모의고사에서 만점을 받는 것과 똑같습니다. 그 만점은 실력이 아닙니다.

**데이터 누수(Data Leakage)** 는 머신러닝 실무에서 평가를 낙관적으로 만드는 주요 위험 가운데 하나입니다. 흔한 이유는 코드가 아무 오류 없이 잘 돌아가기 때문입니다. 오히려 점수가 **올라갑니다.** 그래서 발견하기 어렵습니다.

> ❓ **질문하기**  
> 답이 새어 들어가는 경로는 어디이고, 어떻게 알아차리는가?

## 💡 쉽게 말하면 — 채점하기 전에 답을 흘리기

시험을 공정하게 치르려면 **문제와 답이 학생에게 미리 가면 안 됩니다.** 그런데 답이 새는 경로는 생각보다 다양합니다.

- 시험 문제를 미리 보여준 경우 (테스트 데이터로 학습)
- 답지를 보고 힌트를 만들어 배포한 경우 (**타깃 정보로 피처를 만듦**)
- 여러 번 시험 보고 가장 잘 본 것만 성적으로 인정한 경우 (**테스트로 설정 고르기**)
- 시험 범위를 답지 기준으로 정한 경우 (전체 데이터로 전처리 기준 계산)

머신러닝의 누수도 이와 비슷한 경로로 발생합니다. 그리고 공통점이 있습니다 — **평가에 쓸 정보가 학습이나 선택에 미리 관여했다**는 것입니다.

```text
[정상]   학습 조각 ──fit──> 기준값·모델 ──apply──> 검증 조각 ──> 정직한 점수
                                                    ↑
                                          검증 조각은 아무것도 기여하지 않음

[누수]   전체 데이터 ──fit──> 기준값·모델 ──apply──> 검증 조각 ──> 부풀려진 점수
            ↑
     검증 조각도 기준값 계산에 기여했다 (= 답이 샜다)
```

## 🔍 자세히 알아보기 — 누수의 네 가지 유형

### 유형 ① 전처리 누수 (가장 흔함)

전처리를 **전체 데이터에 먼저** 적용하고 그다음 나눕니다.

```python
# ✗ 누수
X_scaled = StandardScaler().fit_transform(X)      # 전체로 fit
cross_val_score(model, X_scaled, y, cv=5)         # 이미 늦었다
```

검증 조각의 평균·표준편차가 스케일러에 들어갔습니다. 결측 대치의 중앙값도 마찬가지입니다.

### 유형 ② 타깃·시점 누수

**타깃 정보나 예측 시점 이후의 정보가 피처에 들어갑니다.** 타깃 인코딩은 범주별 타깃 평균을 사용하므로, 학습 행 자체의 타깃이 인코딩값에 섞이지 않도록 교차 적합(Cross-Fitting)이 필요합니다.

또 다른 형태는 **예측 시점에 없는 정보**를 쓰는 것입니다. "해지 사유" 컬럼으로 해지를 예측하는 식입니다.

### 유형 ③ 검증 누수 (알아차리기 어려움)

**테스트 점수를 보고 설정을 고릅니다.** 여러 `max_depth`를 시도해 테스트 점수가 가장 좋은 것을 채택하고, 그 테스트 점수를 최종 성능으로 보고하는 경우입니다. 테스트셋이 선택에 관여했으므로 그 점수는 더 이상 "처음 보는 데이터에서의 성능"이 아닙니다.

### 유형 ④ 시간·중복 누수

시계열에서 **미래 데이터가 과거 학습에 섞이거나**, 같은 사람의 기록이 학습과 검증에 나뉘어 들어가는 경우입니다.

| 유형        | 무엇이 새는가                 | 처방                                              |
| ----------- | ----------------------------- | ------------------------------------------------- |
| ① 전처리    | 검증 조각의 분포 정보         | `Pipeline` (Part 3)                               |
| ② 타깃·시점 | 타깃 또는 예측 시점 이후 정보 | 교차 적합 인코더 · 피처 생성 시점 점검            |
| ③ 검증      | 테스트셋이 선택에 관여        | 검증셋·교차 검증에서 고르고 테스트는 마지막 한 번 |
| ④ 시간·중복 | 미래 정보 · 같은 개체         | `TimeSeriesSplit` · `GroupKFold`                  |

> ⚠️ **주의하기**  
> 검증 점수가 업무 기준이나 단순 기준 모델보다 예상 밖으로 높다면 피처 출처와 분할 절차를 점검합니다. 작은 CV 표준편차는 모델이 안정적일 때도 나타나므로 누수의 증거로 사용할 수 없습니다.

> 📌 **실무 연결하기**  
> 누수 사고는 산업마다 얼굴이 다릅니다 — **금융**에서 부도 후 발생한 연체 기록을 피처로 쓰거나, **헬스케어**에서 진단 후 처방된 약물 정보가 섞이거나, **제조**에서 불량 판정 후의 재검사 데이터가 들어가거나, **이커머스**에서 구매 후 리뷰 작성 여부로 구매를 예측하는 식입니다. 전부 **"결과가 나온 뒤에 생긴 정보"** 라는 공통점이 있습니다.

## 📊 데이터로 확인해 봅시다 — 검증 누수를 재현하기

먼저 유형 ③ **검증 누수**를 봅니다. 이것은 `Pipeline`으로도 막을 수 없는 유형이라, 오늘 배울 것 중 유일하게 **사람의 절차**로만 막을 수 있습니다.

의사결정나무의 `max_depth`를 고른다고 해봅시다. 두 가지 방법이 있습니다.

- **(나쁜 방법)** 여러 깊이로 학습해 **테스트 점수**가 가장 높은 것을 고르고, 그 점수를 최종 성능으로 보고합니다.
- **(옳은 방법)** 학습 데이터 안에서 **교차 검증 점수**가 가장 높은 것을 고르고, 테스트는 마지막에 한 번만

> 🤔 **예상하기**  
> 두 방법이 같은 깊이를 고를지 예상합니다. 다른 깊이를 고른다면 같은 테스트셋에서 보고되는 점수 차이가 어느 정도일지 확인합니다.

▶️ **코드 실행하기 · 코드 셀 7 [C7]**

In [7]:
# ─────────────────────────────────────────────
# [C7] Part 2 · 📊 데이터로 확인해 봅시다 — 검증 누수를 재현하기
# (나쁜 방법) 테스트로 고르기 vs (옳은 방법) CV로 고르기 (실행하면 두 결론이 비교됩니다)
# ─────────────────────────────────────────────
FEAT_NUM = NUM_COLS                       # [C5]에서 정한 수치형 8개
FEAT_CAT = ["sex", "embarked", "who"]
X = df[FEAT_NUM + FEAT_CAT]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)


def make_pre():
    """수치·범주 분기 전처리를 새로 만들어 돌려줍니다."""
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), FEAT_NUM),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]), FEAT_CAT),
    ])


rows = []
for depth in [2, 3, 4, 5, 7, 10, None]:
    pipe = Pipeline([("pre", make_pre()),
                     ("clf", DecisionTreeClassifier(max_depth=depth,
                                                    random_state=RANDOM_STATE))])
    cv_score = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy").mean()
    pipe.fit(X_train, y_train)
    test_score = pipe.score(X_test, y_test)
    rows.append({"max_depth": "제한 없음" if depth is None else depth,
                 "학습셋 CV": cv_score, "테스트": test_score})

sel = pd.DataFrame(rows)
print(sel.round(4).to_string(index=False))
print()
bad = sel.loc[sel["테스트"].idxmax()]
good = sel.loc[sel["학습셋 CV"].idxmax()]
print(f"✗ 나쁜 방법 — 테스트로 고름: max_depth={bad['max_depth']} → 보고 점수 {bad['테스트']:.4f}")
print(f"✓ 옳은 방법 — CV로 고름:    max_depth={good['max_depth']} → 테스트 점수 {good['테스트']:.4f}")
print(f"→ 부풀림 {bad['테스트'] - good['테스트']:+.4f}")

max_depth  학습셋 CV    테스트
        2  0.7949 0.7937
        3  0.8113 0.8072
        4  0.8053 0.8161
        5  0.8158 0.7623
        7  0.8069 0.7578
       10  0.7814 0.7489
    제한 없음  0.7620 0.7265

✗ 나쁜 방법 — 테스트로 고름: max_depth=4 → 보고 점수 0.8161
✓ 옳은 방법 — CV로 고름:    max_depth=5 → 테스트 점수 0.7623
→ 부풀림 +0.0538


> 🎯 **[C7] 확인하기**  
> 두 방법이 **다른 깊이**를 고릅니다.

| `max_depth` | 학습셋 CV            | 테스트                   |
| ----------- | -------------------- | ------------------------ |
| 2           | 0.7949               | 0.7937                   |
| 3           | 0.8113               | 0.8072                   |
| **4**       | 0.8053               | **0.8161** ← 테스트 최고 |
| **5**       | **0.8158** ← CV 최고 | 0.7623                   |
| 7           | 0.8069               | 0.7578                   |
| 10          | 0.7814               | 0.7489                   |
| 제한 없음   | 0.7620               | 0.7265                   |

- **나쁜 방법**은 `max_depth=4`를 골라 **0.8161**을 보고합니다.
- **옳은 방법**은 `max_depth=5`를 골라 **0.7623**을 보고합니다.

같은 테스트셋에서 보고되는 차이는 **+0.0538**입니다. 테스트셋으로 후보를 선택한 값이 CV로 선택한 절차의 테스트 점수보다 5.4%p 높게 나타납니다. 이 차이를 모집단 일반화 성능의 실제 부풀림으로 단정할 수는 없습니다.

여기서 핵심은 **어느 쪽 숫자가 진실에 가까운가**입니다. `max_depth=4`가 테스트에서 0.8161을 낸 것은 사실이지만, **그 사실을 알고 4를 골랐기 때문에** 0.8161은 더 이상 "처음 보는 데이터에서의 성능"이 아닙니다. 테스트셋이 이미 선택에 관여했으니까요.

CV로 설정을 고른 뒤 처음 한 번 확인한 테스트 점수 0.7623은 모델 선택에 사용되지 않은 추정값입니다. 다만 하나의 테스트 분할도 표본 변동이 있으므로 새로운 데이터에서 같은 값이 그대로 재현된다고 보장할 수는 없습니다.

표를 자세히 보면 흥미로운 점이 있습니다 — **CV 최고(5)와 테스트 최고(4)가 다르고, 두 열의 순위가 전반적으로 어긋납니다.** 이것이 바로 "테스트셋으로 후보를 고르면 해당 분할에 과적합될 수 있다"는 뜻입니다. CV는 여러 번 나눠 재므로 특정 분할의 운에 덜 흔들립니다.

> 💡 **핵심짚기**  
> 이 누수는 코드로 막을 수 없습니다. **절차로만 막힙니다** — 설정은 학습 데이터 안(검증셋 또는 교차 검증)에서 고르고, 테스트셋은 **최종 확인 한 번**만 쓰는 것입니다. 여러 번 테스트를 보면 그것도 결국 학습에 쓴 셈이 됩니다.

## ⌨️ 백문이 불여일타! (2)

```
[문제]
아래는 동료가 짠 코드입니다. 누수가 몇 개 있는지 찾습니다.

  # 1) 전체 데이터로 결측을 채운다
  X_filled = X.copy()
  X_filled["age"] = X_filled["age"].fillna(X_filled["age"].median())

  # 2) 전체 데이터로 스케일링한다
  X_scaled = StandardScaler().fit_transform(X_filled[FEAT_NUM])

  # 3) 나눠서 학습한다
  Xa, Xb, ya, yb = train_test_split(X_scaled, y, test_size=0.25, random_state=42)
  model = DecisionTreeClassifier(max_depth=4).fit(Xa, ya)
  print("테스트 정확도:", model.score(Xb, yb))

1) 누수가 발생한 줄 번호와, 각각 무엇이 새는지 적습니다.
2) 이 코드의 "테스트 정확도"를 신뢰할 수 있습니까? 이유와 함께 적습니다.
3) 유형 ①~④ 중 이 코드에 해당하는 것은 무엇입니까?
```

▶️ **코드 실행하기 · 코드 셀 8 [C8]**

In [10]:
# [C8] Part 2 · ⌨️ 백문이 불여일타! (2)
# ⌨️ 백문이 불여일타! (2) — 동료의 코드에서 누수 찾기

# (답을 코드로 검증해보고 싶다면 아래에 직접 재현해보세요)

# [C8] Part 2 · ⌨️ 백문이 불여일타! (2)
# 백문이 불여일타! (2) — 동료의 코드에서 누수 찾기

# 1) 동료의 코드를 그대로 재현
X_filled = X.copy()
X_filled["age"] = X_filled["age"].fillna(X_filled["age"].median())  # 전체로 median 계산

X_scaled = StandardScaler().fit_transform(X_filled[FEAT_NUM])       # 전체로 fit

Xa, Xb, ya, yb = train_test_split(X_scaled, y, test_size=0.25, random_state=42)
model = DecisionTreeClassifier(max_depth=4).fit(Xa, ya)
leak_score = model.score(Xb, yb)
print("동료 코드 (누수 있음) 테스트 정확도:", round(leak_score, 4))

print()

# 2) 올바른 방법 — Pipeline으로 학습 조각에만 fit
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size=0.25, random_state=42)

clean_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", DecisionTreeClassifier(max_depth=4, random_state=42)),
])
clean_pipe.fit(X_train2[FEAT_NUM], y_train2)
clean_score = clean_pipe.score(X_test2[FEAT_NUM], y_test2)
print("Pipeline (누수 없음) 테스트 정확도:", round(clean_score, 4))

print()
print(f"→ 차이 {leak_score - clean_score:+.4f}")

동료 코드 (누수 있음) 테스트 정확도: 0.704

Pipeline (누수 없음) 테스트 정확도: 0.704

→ 차이 +0.0000


<details>
<summary>(클릭) 💡 힌트</summary>

- 각 줄에서 **기준값이 어느 데이터로부터 계산되는지** 따라갑니다 — 중앙값, 평균, 표준편차입니다.
- `train_test_split`이 **몇 번째 줄에** 있는지가 결정적입니다.
- 3번은 §🔍의 유형 표를 확인합니다. 이 코드는 하이퍼파라미터를 고르지도, 타깃으로 피처를 만들지도 않습니다.

</details>

## 🚦 짚고 넘어가기

1. 누수의 네 가지 유형을 각각 한 줄로 설명할 수 있습니까?
2. 누수를 의심해야 하는 신호는 무엇입니까?
3. 검증 누수(유형 ③)가 `Pipeline`으로 막히지 않는 이유는 무엇입니까?

> ⏭️ **다음 학습 예고**  
> 누수의 지도를 그렸습니다. 유형 ①은 적절한 변환기를 `Pipeline` 안에 두어 예방할 수 있습니다. 유형 ②는 타깃을 안전하게 다루는 변환기와 예측 시점 점검이 함께 필요합니다. 코드가 길어질수록 전처리 순서를 놓치기 쉬우며, AI가 제안한 코드도 같은 검토가 필요합니다. 다음 Part에서는 전처리의 `fit` 범위를 학습 조각으로 고정하는 구조를 배웁니다. 그리고 타깃 누수의 부풀림이 얼마나 큰지 직접 재봅니다.

# Part 3. `Pipeline` — 전처리 누수를 구조로 예방한다

Part 2에서 누수의 지도를 그렸습니다. 그러면 처방은 무엇일까요?

전처리 순서를 사람의 기억에만 맡기면 재현하기 어렵습니다. 이유는 세 가지입니다.

1. 코드가 길어지면 전처리 순서를 놓칠 가능성이 커집니다
2. 교차 검증은 데이터를 5번 다르게 나누므로, 5번 각각에 대해 전처리를 따로 해야 합니다 — 손으로는 사실상 불가능합니다
3. AI가 제안한 코드도 `fit` 범위를 별도로 확인해야 합니다

그래서 scikit-learn은 다른 접근을 택했습니다. **전처리와 모델의 학습 범위를 하나의 객체로 관리하는 것**입니다.

> ❓ **질문하기**  
> 전처리와 모델을 하나로 묶으면 무엇이 구조적으로 불가능해지는가?

## 💡 쉽게 말하면 — 컨베이어 벨트로 묶기

공장에서 부품을 손으로 옮기면 순서가 뒤바뀌거나 한 단계를 빼먹을 수 있습니다. **컨베이어 벨트에 올려놓으면** 순서가 물리적으로 고정됩니다. 빼먹을 수가 없습니다.

`Pipeline`이 그 벨트입니다. "결측 대치 → 스케일링 → 모델"을 한 객체로 묶으면, 그 객체는 **하나의 모델처럼** 행동합니다.

```text
[손으로 하면]
  imputer.fit(???)   ← 무엇에 fit할지 매번 사람이 결정 = 실수 지점
  scaler.fit(???)    ← 또 결정
  model.fit(???)     ← 또 결정
  ... 교차 검증 5겹이면 이 결정을 15번 해야 한다

[Pipeline으로 묶으면]
  pipe = Pipeline([("impute", ...), ("scale", ...), ("clf", ...)])
  pipe.fit(주어진 데이터)     ← 결정 지점이 하나
                              그리고 그 하나를 cross_val_score가 알아서 학습 조각에만 준다
```

핵심은 마지막 줄입니다. **`Pipeline`은 자신에게 주어진 데이터에만 `fit`합니다.** 교차 검증이 fold를 나눠 넘기면 전처리의 `fit`도 자동으로 그 학습 조각에만 일어납니다. 사람이 개입할 틈이 없어집니다.

## 🔍 자세히 알아보기 — 두 개의 도구, 두 겹의 묶음

### `Pipeline` — 단계를 순서대로 잇는다

```python
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression()),
])
```

- 각 단계는 `("이름", 객체)` 쌍입니다. 이름은 나중에 튜닝할 때 씁니다
- **마지막 단계만 모델**이고 앞의 단계는 모두 변환기입니다
- `pipe.fit(X, y)` → 앞 단계들을 차례로 `fit_transform`하고 마지막에 모델을 `fit`합니다
- `pipe.predict(X_new)` → 앞 단계들은 `transform`만 하고(학습 때의 기준값 사용) 모델이 예측합니다

**`fit`과 `transform`이 분리된 것**이 핵심입니다. 학습 때 계산한 중앙값·평균을 새 데이터에 **적용만** 합니다.

### `ColumnTransformer` — 열마다 다른 처리를 한다

문제가 하나 있습니다. `Pipeline`은 모든 열에 같은 처리를 합니다. 그런데 우리 데이터는 수치형과 범주형이 섞여 있고, 둘은 다른 전처리가 필요합니다.

```python
pre = ColumnTransformer([
    ("num", 수치용_Pipeline, ["age", "fare", ...]),
    ("cat", 범주용_Pipeline, ["sex", "embarked", ...]),
])
```

`ColumnTransformer`는 **열 그룹마다 다른 변환기를 배정**하고 결과를 옆으로 이어 붙입니다. 그리고 이것 자체가 하나의 변환기이므로, 다시 `Pipeline`에 넣을 수 있습니다.

```text
최종 구조 = 두 겹의 묶음

Pipeline
 ├─ ("pre", ColumnTransformer)
 │        ├─ ("num", Pipeline[대치 → 표준화])  →  수치형 열들
 │        └─ ("cat", Pipeline[대치 → One-Hot]) →  범주형 열들
 └─ ("clf", LogisticRegression)
```

### 유용한 옵션 두 개

- **`remainder`** — 지정하지 않은 열의 처리. 기본은 `"drop"`(버림), `"passthrough"`면 그대로 통과
- **`get_feature_names_out()`** — 변환 후 피처 이름. One-Hot으로 펼쳐진 뒤 어느 열이 무엇인지 알려줍니다. 나중에 해석할 때 필수입니다

> ⚠️ **주의하기**  
> `Pipeline`은 그 안에 포함된 변환기의 `fit` 범위를 학습 조각으로 제한해 **전처리 누수**를 예방합니다. `TargetEncoder`처럼 `fit_transform`에서 교차 적합을 수행하는 변환기를 넣으면 타깃 인코딩 누수도 줄일 수 있습니다. 예측 시점 이후 피처는 `Pipeline`만으로 식별할 수 없고, 유형 ③(검증 누수)은 평가 절차로, 유형 ④(시간·중복)는 `TimeSeriesSplit`·`GroupKFold` 같은 분할기로 다룹니다. `Pipeline`을 사용해도 전체 누수 감사를 생략할 수 없습니다.

> 📌 **실무 연결하기**  
> `Pipeline`의 진짜 값어치는 누수 방지 하나가 아닙니다. **배포 시점의 일관성**이 두 번째 값어치입니다 — 저장된 `Pipeline`은 전처리 기준값까지 품고 있어서, 3개월 뒤 새 데이터가 와도 학습 때와 똑같은 변환이 적용됩니다. 전처리 코드를 따로 관리하면 학습과 운영의 설정이 어긋날 위험이 커집니다.

## 📊 데이터로 확인해 봅시다 — 묶고, 새는지 재보기

먼저 기본 `Pipeline`을 만들고 내부를 들여다봅니다.

> 🤔 **예상하기**  
> `Pipeline`으로 묶은 객체에 `.fit()`을 부르면 내부에서 몇 번의 `fit`이 일어날까요? 그리고 `.predict()`를 부를 때 전처리 단계는 `fit`을 다시 할까요, 안 할까요?

▶️ **코드 실행하기 · 코드 셀 9 [C9]**

In [ ]:
# ─────────────────────────────────────────────
# [C9] Part 3 · 📊 데이터로 확인해 봅시다 — 묶고, 새는지 재보기
# 기본 Pipeline 만들기 → 내부 단계 확인 (실행하면 단계 목록과 학습된 기준값이 나옵니다)
# ─────────────────────────────────────────────
simple = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

simple.fit(X_train[FEAT_NUM], y_train)

print("Pipeline 단계 목록")
for name, step in simple.named_steps.items():
    print(f"  {name:8} {type(step).__name__}")
print()
print("named_steps로 내부에 접근할 수 있습니다")
med = simple.named_steps["impute"].statistics_
print(f"  대치에 쓴 중앙값 (앞 3개): {med[:3].round(2)}")
print(f"  → age 중앙값 {med[FEAT_NUM.index('age')]:.1f}  (학습 조각에서만 계산된 값)")
print()
print(f"테스트 정확도: {simple.score(X_test[FEAT_NUM], y_test):.4f}")
print()
print("→ predict할 때 전처리는 fit을 다시 하지 않고, 학습 때의 기준값을 '적용만' 합니다.")

Pipeline 단계 목록
  impute   SimpleImputer
  scale    StandardScaler
  clf      LogisticRegression

named_steps로 내부에 접근할 수 있습니다
  대치에 쓴 중앙값 (앞 3개): [ 3. 29.  0.]
  → age 중앙값 29.0  (학습 조각에서만 계산된 값)

테스트 정확도: 0.6816

→ predict할 때 전처리는 fit을 다시 하지 않고, 학습 때의 기준값을 '적용만' 합니다.

> 🎯 **[C9] 확인하기**  
> `.fit()` 한 번에 내부 세 단계가 순서대로 적합됩니다 — 대치용 중앙값 계산, 표준화용 평균·표준편차 계산, 그리고 모델 학습입니다.

그리고 `.predict()`에서는 전처리가 **`fit`을 다시 하지 않습니다.** `transform`만 합니다. 학습 때 계산한 중앙값을 테스트 데이터에 **적용만** 하는 것입니다. 이것이 누수를 막는 메커니즘의 핵심입니다.

`named_steps`로 내부를 꺼내 볼 수 있다는 것도 기억합니다. 디버깅할 때, 그리고 나중에 SHAP으로 해석할 때 씁니다.

이제 열마다 다른 처리를 하는 `ColumnTransformer`를 붙입니다.

▶️ **코드 실행하기 · 코드 셀 10 [C10]**

In [ ]:
# ─────────────────────────────────────────────
# [C10] Part 3 · 📊 데이터로 확인해 봅시다 — 묶고, 새는지 재보기
# ColumnTransformer로 수치·범주 분기 → 두 겹 Pipeline 완성 (실행하면 변환 후 피처명이 나옵니다)
# ─────────────────────────────────────────────
model = Pipeline([("pre", make_pre()),
                  ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])
model.fit(X_train, y_train)

names = model.named_steps["pre"].get_feature_names_out()
print(f"입력 컬럼 {X_train.shape[1]}개 → 변환 후 피처 {len(names)}개")
print()
print("변환 후 피처 이름")
for n in names:
    print(f"  {n}")
print()
print(f"테스트 정확도: {model.score(X_test, y_test):.4f}")
print()
print("→ 'num__'·'cat__' 접두사로 어느 갈래에서 왔는지 알 수 있습니다.")
print("  범주형 3개가 One-Hot으로 펼쳐져 열이 늘어났습니다.")

입력 컬럼 11개 → 변환 후 피처 16개

변환 후 피처 이름
  num__pclass
  num__age
  num__sibsp
  num__parch
  num__fare
  num__family_size
  num__is_alone
  num__fare_per_person
  cat__sex_female
  cat__sex_male
  cat__embarked_C
  cat__embarked_Q
  cat__embarked_S
  cat__who_child
  cat__who_man
  cat__who_woman

테스트 정확도: 0.8161

→ 'num__'·'cat__' 접두사로 어느 갈래에서 왔는지 알 수 있습니다.
  범주형 3개가 One-Hot으로 펼쳐져 열이 늘어났습니다.

> 👀 **[C10] 결과읽기**  
> 입력 11개 컬럼이 변환 후 더 많은 피처가 됐습니다. `sex`(2) + `embarked`(3) + `who`(3)가 One-Hot으로 펼쳐진 결과입니다.

피처 이름에 붙은 **`num__`·`cat__` 접두사**가 어느 갈래에서 왔는지 알려줍니다. 이 이름이 있어야 나중에 "어떤 피처가 중요했나"를 사람이 읽을 수 있습니다.

이제 오늘의 핵심 실험입니다. **누수 버전과 `Pipeline` 버전의 점수를 직접 비교**합니다.

동료가 이런 피처를 만들었다고 해봅시다 — **"운임과 등급 조합별 과거 생존율"**. 실무에서는 "고객 세그먼트별 과거 이탈률", "지역별 부도율" 같은 형태로 아주 흔하게 등장하는 피처입니다. 좋은 아이디어처럼 보입니다.

▶️ **코드 실행하기 · 코드 셀 11 [C11]**

In [ ]:
# ─────────────────────────────────────────────
# [C11] Part 3 · 📊 데이터로 확인해 봅시다 — 묶고, 새는지 재보기
# 타깃 누수 재현 — "세그먼트별 과거 생존율" 피처 (실행하면 부풀림 크기가 나옵니다)
# ─────────────────────────────────────────────
# 동료가 만든 세그먼트 키 (운임 + 등급 조합)
seg = df["fare"].round(2).astype(str) + "_" + df["pclass"].astype(str)
print(f"세그먼트 키의 고유값: {seg.nunique()}개 / 전체 {len(seg)}명")
print(f"  → 세그먼트 하나에 평균 {len(seg) / seg.nunique():.1f}명뿐입니다. 이것이 문제의 씨앗입니다.")
print()

# ✗ 누수 버전 — 전체 데이터의 타깃으로 세그먼트별 생존율을 계산해 피처로 씀
seg_rate = y.groupby(seg).mean()            # 전체 y를 봤다!
df_leak = df.copy()
df_leak["seg_survival_rate"] = seg.map(seg_rate)

LEAK_NUM = FEAT_NUM + ["seg_survival_rate"]
pre_leak = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), LEAK_NUM),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OneHotEncoder(handle_unknown="ignore"))]), FEAT_CAT),
])
leaky = cross_val_score(
    Pipeline([("pre", pre_leak), ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
    df_leak[LEAK_NUM + FEAT_CAT], y, cv=cv, scoring="accuracy")

# ✓ Pipeline 버전 — 같은 아이디어를 TargetEncoder로, Pipeline 안에서 (조각별로 따로 계산)
df_ok = df.copy()
df_ok["segment"] = seg
pre_ok = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), FEAT_NUM),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OneHotEncoder(handle_unknown="ignore"))]), FEAT_CAT),
    ("seg", TargetEncoder(random_state=RANDOM_STATE), ["segment"]),
])
clean = cross_val_score(
    Pipeline([("pre", pre_ok), ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
    df_ok[FEAT_NUM + FEAT_CAT + ["segment"]], y, cv=cv, scoring="accuracy")

print(f"✗ 누수 버전 (전체 y로 생존율 계산)  CV {leaky.mean():.4f} ± {leaky.std():.4f}")
print(f"✓ Pipeline 버전 (조각별로 따로 계산) CV {clean.mean():.4f} ± {clean.std():.4f}")
print()
print(f"→ 부풀림 {leaky.mean() - clean.mean():+.4f}")
print(f"  누수 버전의 표준편차({leaky.std():.4f})가 유난히 작은 것도 누수 신호입니다.")

세그먼트 키의 고유값: 243개 / 전체 891명
  → 세그먼트 하나에 평균 3.7명뿐입니다. 이것이 문제의 씨앗입니다.

✗ 누수 버전 (전체 y로 생존율 계산)  CV 0.8945 ± 0.0074
✓ Pipeline 버전 (조각별로 따로 계산) CV 0.8350 ± 0.0165

→ 부풀림 +0.0595
  누수 버전의 표준편차(0.0074)가 유난히 작은 것도 누수 신호입니다.

> 👀 **[C11] 결과읽기**  
> 부풀림이 **+0.0595**입니다. 약 6%p입니다.

| 버전                              | CV 정확도           |
| --------------------------------- | ------------------- |
| ✗ 누수 (전체 `y`로 생존율 계산)   | **0.8945 ± 0.0074** |
| ✓ `Pipeline` (조각별로 따로 계산) | 0.8350 ± 0.0165     |

왜 이렇게 크게 새는지 확인합니다. 세그먼트 키의 고유값이 **243개**인데 전체가 891명입니다. 세그먼트 하나에 평균 **3.7명**뿐입니다. 그러면 "이 세그먼트의 생존율"은 사실상 **그 3~4명의 정답 평균**이고, 각 사람의 피처에 **자기 자신의 정답이 들어갑니다.**

모델은 이것을 놓치지 않습니다. `seg_survival_rate`만 보면 정답을 거의 알 수 있으니, 0.8945라는 훌륭한 점수가 나옵니다. 그리고 이 점수는 검증 행의 타깃이 피처 계산에 참여해 낙관적으로 편향되었습니다. 운영 데이터에서는 같은 방식으로 미래 타깃을 사용할 수 없으므로 개발 점수가 재현되지 않을 수 있습니다.

누수 버전의 겹별 표준편차는 **±0.0074**, 정상 버전은 **±0.0165**입니다. 이 사례에서는 누수 버전의 변동이 더 작지만, 작은 표준편차만으로 누수를 판정할 수는 없습니다.

`TargetEncoder`를 `Pipeline` 안에 넣으면 바깥쪽 교차 검증의 각 학습 조각에서 `fit_transform`이 내부 교차 적합을 수행합니다. 검증 조각에는 학습 조각에서 얻은 인코딩만 적용되어 0.8350을 냅니다. 이 값이 정직한 추정입니다.

> 💡 **핵심짚기**  
> 같은 피처 아이디어인데 **어디서 계산하느냐**가 0.060의 차이를 만들었습니다. 아이디어가 나쁜 것이 아니라 **계산 위치가 잘못된 것**입니다. `Pipeline` 안으로 옮기면 같은 아이디어가 정직해집니다.

그런데 여기서 정직하게 짚어야 할 것이 있습니다. **모든 누수가 이렇게 크게 부풀려지는 것은 아닙니다.**

▶️ **코드 실행하기 · 코드 셀 12 [C12]**

In [ ]:
# ─────────────────────────────────────────────
# [C12] Part 3 · 📊 데이터로 확인해 봅시다 — 묶고, 새는지 재보기
# 전처리 누수(유형 ①)는 얼마나 부풀리나 — 정직한 재검증 (실행하면 차이가 나옵니다)
# ─────────────────────────────────────────────
# ✗ 누수 버전 — 대치와 표준화를 전체 데이터로 먼저
X_leaked = make_pre().fit_transform(X)      # 전체 X로 fit (검증 조각 포함)
leak_pre = cross_val_score(
    LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    X_leaked, y, cv=cv, scoring="accuracy")

# ✓ Pipeline 버전
clean_pre = cross_val_score(
    Pipeline([("pre", make_pre()),
              ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
    X, y, cv=cv, scoring="accuracy")

print("누수 유형별 부풀림 크기 비교")
print(f"  유형 ② 타깃 누수   : {leaky.mean() - clean.mean():+.4f}   ([C11] 결과)")
print(f"  유형 ① 전처리 누수 : {leak_pre.mean() - clean_pre.mean():+.4f}")
print()
print(f"    누수 버전   CV {leak_pre.mean():.4f} ± {leak_pre.std():.4f}")
print(f"    Pipeline    CV {clean_pre.mean():.4f} ± {clean_pre.std():.4f}")
print()
print("→ 전처리 누수는 이 데이터에서 거의 부풀리지 않았습니다. 그래도 원칙은 같습니다.")

누수 유형별 부풀림 크기 비교
  유형 ② 타깃 누수   : +0.0595   ([C11] 결과)
  유형 ① 전처리 누수 : +0.0000

    누수 버전   CV 0.8204 ± 0.0080
    Pipeline    CV 0.8204 ± 0.0080

→ 전처리 누수는 이 데이터에서 거의 부풀리지 않았습니다. 그래도 원칙은 같습니다.

> 👀 **[C12] 결과읽기**  
> 전처리 누수의 부풀림은 **+0.0000**입니다. 소수점 넷째 자리까지 같습니다.

| 누수 유형          | 부풀림      |
| ------------------ | ----------- |
| 유형 ② 타깃 누수   | **+0.0595** |
| 유형 ① 전처리 누수 | **+0.0000** |

같은 '누수'인데 한쪽은 6%p, 한쪽은 0입니다. 왜일까요?

**전처리 누수가 새는 것은 "분포 통계량"뿐**입니다. 891행에서 전체의 중앙값과 학습 조각(713행)의 중앙값은 거의 같습니다. 새는 정보의 양이 미미합니다.

**타깃 누수가 새는 것은 "정답 그 자체"** 입니다. 양이 아니라 종류가 다릅니다.

그러면 전처리 누수는 신경 쓰지 않아도 될까요? **아닙니다.** 세 가지 이유가 있습니다.

1. **표본이 적으면 벌어집니다.** 891행이라 통계량이 안정적이었을 뿐입니다
2. **강한 변환일수록 벌어집니다.** 전체 데이터로 수행한 피처 선택·차원 축소는 더 큰 편향을 만들 수 있습니다. 타깃 인코딩은 별도의 타깃 누수 위험이며 [C11]이 그 예입니다
3. `Pipeline`은 전처리 순서와 적용 범위를 한 객체에 기록해 코드 중복과 운영 불일치도 줄입니다

> 💡 **핵심짚기**  
> _"누수는 발생 여부를 매번 확인하는 것이 아니라 구조로 차단한다."_ 오늘 우리는 부풀림이 0.060인 누수와 0.000인 누수를 모두 봤습니다. 부풀림의 크기와 관계없이 평가 절차를 일관되게 유지하기 위해 학습되는 전처리를 `Pipeline` 안에 둡니다.

## ⌨️ 백문이 불여일타! (3)

```
[문제]
Part 2의 ⌨️(2)에서 본 동료의 누수 코드를, 이제 Pipeline으로 고칩니다.

1) 결측 대치(중앙값) → 표준화 → DecisionTreeClassifier(max_depth=4)를
   하나의 Pipeline으로 만듭니다. (수치형 FEAT_NUM만 사용)
2) 그 Pipeline으로 계층 5겹 교차 검증 점수를 구합니다.
3) named_steps로 대치에 쓴 age 중앙값을 꺼내 출력합니다.
   전체 데이터의 age 중앙값과 비교하면 어떻습니까?
```

▶️ **코드 실행하기 · 코드 셀 13 [C13]**

In [ ]:
# [C13] Part 3 · ⌨️ 백문이 불여일타! (3)
# ⌨️ 백문이 불여일타! (3) — 누수 코드를 Pipeline으로 고치기

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- `Pipeline([("impute", ...), ("scale", ...), ("clf", ...)])` 형태로 세 단계를 잇습니다.
- 교차 검증은 `cross_val_score(pipe, df[FEAT_NUM], y, cv=cv, scoring="accuracy")`입니다.
- 3번은 `pipe.fit(...)`을 먼저 해야 `statistics_`가 생깁니다. 전체 중앙값은 `df["age"].median()`입니다.

</details>

## 🚦 짚고 넘어가기

1. `Pipeline`이 누수를 막는 메커니즘을 한 문장으로 설명할 수 있습니까?
2. `ColumnTransformer`가 필요한 이유는 무엇입니까?
3. 타깃 누수(+0.0595)와 전처리 누수(+0.000)의 부풀림 차이가 왜 그렇게 큽니까?

> ⏭️ **다음 학습 예고**  
> 학습되는 전처리를 한 객체로 묶었습니다. 이제 그 파이프 **안에서** 실험할 수 있습니다 — 전처리 옵션과 모델 설정을 함께 튜닝하고, 그 결과를 기록으로 남기고, 최종 모델을 저장합니다. 그리고 마지막에 이 데이터에서 다음 실험의 우선순위를 정할 숫자를 봅니다 — **어떤 레버를 먼저 당겨야 하는가**에 대한 답입니다.

# Part 4. 평가 정보를 분리한 실험 — 튜닝·기록·저장

`Pipeline`을 갖췄으니 이제 실험을 할 수 있습니다. 그런데 이 Part의 제목이 "튜닝"이 아니라 **"실험"** 인 이유가 있습니다.

탐색 계산은 도구가 수행하지만, 사람이 정해야 할 것은 **어떤 실험을 설계하고 그 결과를 어떻게 기록해 결정으로 바꾸느냐**로 옮겨갔습니다.

> ❓ **질문하기**  
> 전처리 기준이 검증 조각에 섞이지 않게 실험하고, 그 결과를 결정으로 바꾸는 절차는 무엇인가?

## 💡 쉽게 말하면 — 실험 노트를 쓰는 과학자

과학자가 실험을 하면 반드시 노트를 씁니다. **무슨 조건으로, 무슨 결과가 나왔고, 그래서 무엇을 결정했는지**를 적습니다.

기록이 없으면 시간이 지난 뒤 설정을 선택한 근거를 추적하기 어렵고, 이미 기각한 실험을 반복할 수 있습니다.

머신러닝 실험도 같습니다. 그래서 오늘의 산출물은 최적 하이퍼파라미터가 아니라 **실험 기록표**입니다.

| 조건        | 지표      | **결정**         |
| ----------- | --------- | ---------------- |
| 원본 피처만 | CV 0.7935 | 기준으로 채택    |
| + 파생 3개  | CV 0.8014 | 유지 (+0.008)    |
| 모델 교체   | CV 0.8170 | 기각 (효과 없음) |

세 번째 열이 핵심입니다. 지표만 남기면 숫자는 남지만 **판단의 맥락**이 사라집니다.

## 🔍 자세히 알아보기 — `Pipeline` 통째 튜닝과 레버의 우선순위

### 먼저 이해하기 — `GridSearchCV`란?

모델이 데이터에서 직접 학습하는 계수와 달리, **하이퍼파라미터(Hyperparameter)** 는 학습을 시작하기 전에 사람이 정하는 설정입니다. 로지스틱 회귀의 `C`, 의사결정나무의 `max_depth`, 결측치 대치 방식의 `strategy`가 여기에 해당합니다.

`GridSearchCV`는 후보 설정을 하나씩 손으로 바꾸는 작업을 자동화합니다. 이름 그대로 **후보 격자(Grid)를 모두 탐색(Search)** 하고, 각 후보를 **교차 검증(Cross-Validation)** 으로 비교합니다.

| 구성 요소    | 하는 일                                                        |
| ------------ | -------------------------------------------------------------- |
| `estimator`  | 비교할 모델 또는 `Pipeline`                                    |
| `param_grid` | 시도할 하이퍼파라미터 후보 목록                                |
| `cv`         | 각 후보를 몇 겹 교차 검증으로 평가할지 지정                    |
| `scoring`    | 후보의 우열을 판단할 기준 지표                                 |
| `refit=True` | 가장 높은 평균 점수를 얻은 설정을 전체 입력 데이터로 다시 학습 |

#### 내부에서 실제로 하는 일

1. `param_grid`의 모든 후보 조합을 만듭니다.
2. 각 조합을 동일한 교차 검증 분할에서 평가합니다.
3. `scoring`으로 지정한 지표의 평균이 가장 높은 조합을 선택합니다.
4. 기본 설정인 `refit=True`에서는 선택된 조합을 전체 입력 데이터로 다시 학습해 `best_estimator_`에 저장합니다.

아래 예시는 `C` 4개와 대치 전략 2개를 조합하므로 **4 × 2 = 8개 후보**를 만듭니다. 5겹 교차 검증이면 후보 평가에 8 × 5 = **40번의 학습**이 필요하고, 마지막에 선택된 조합을 전체 데이터로 한 번 더 학습합니다.

| 확인할 속성       | 의미                                             |
| ----------------- | ------------------------------------------------ |
| `best_params_`    | 탐색 범위 안에서 선택된 하이퍼파라미터 조합      |
| `best_score_`     | 선택에 사용된 내부 교차 검증 평균 점수           |
| `best_estimator_` | 선택된 설정으로 다시 학습된 모델 또는 `Pipeline` |
| `cv_results_`     | 모든 후보의 평균 점수·표준편차·순위 등 비교 기록 |

> ⚠️ **구분하기 — 선택 점수와 최종 평가**  
> `best_score_`는 주어진 후보 가운데 하나를 고르는 **모델 선택용 점수**입니다. 탐색에 사용하지 않은 테스트셋이나 바깥쪽 교차 검증으로 확인하기 전에는 최종 일반화 성능으로 보고하지 않습니다. 또한 격자에 넣지 않은 값은 비교하지 않으므로, `best_params_`는 모든 가능한 값 중 절대적인 최적값이 아닙니다.

### `Pipeline`을 그대로 `GridSearchCV`에 넣는다

`Pipeline`은 하나의 모델처럼 행동하므로 그대로 튜닝 대상이 됩니다. 파라미터 이름만 규칙을 따르면 됩니다.

```python
grid = {
    "clf__C": [0.01, 0.1, 1, 10],                    # 모델 파라미터
    "pre__num__impute__strategy": ["median", "mean"], # 전처리 옵션까지!
}
GridSearchCV(pipe, grid, cv=cv, scoring="accuracy")  # 모델 선택용 내부 CV
```

**이름 규칙은 `단계명__파라미터명`** 이고, 중첩되면 `__`로 계속 이어 붙입니다. `pre__num__impute__strategy`는 "`pre` 안의 `num` 안의 `impute`의 `strategy`"입니다.

이것이 `Pipeline`의 세 번째 값어치입니다. **전처리 방식 자체를 튜닝 대상으로 만들 수 있습니다** — 중앙값 대치가 나은지 평균 대치가 나은지를 교차 검증이 정해줍니다. 전처리를 밖에서 미리 적합하면 각 검증 겹의 학습 조각만으로 전처리 옵션을 비교할 수 없습니다.

### 레버의 우선순위 — 무엇을 먼저 당길 것인가

성능을 올리는 레버는 크게 셋입니다.

| 레버              | 하는 일                     | 비용                      |
| ----------------- | --------------------------- | ------------------------- |
| **① 데이터·피처** | 좋은 입력을 만든다          | 도메인 이해 · 사람의 시간 |
| **② 모델**        | 더 표현력 있는 알고리즘으로 | 계산 · 해석 가능성 하락   |
| **③ 튜닝**        | 설정을 최적화한다           | 계산 (조합 수 × 겹 수)    |

일반적으로 문제 정의와 데이터·피처를 먼저 점검한 뒤 모델과 튜닝을 비교합니다. 오늘은 이 데이터와 검증 조건에서 세 레버의 변화량을 확인합니다.

### 저장 — `joblib`

최종 모델은 `joblib`으로 저장합니다. **`Pipeline` 전체를 저장하면 전처리 기준값까지 함께 담깁니다.**

```python
joblib.dump(best_model, "model.joblib")
loaded = joblib.load("model.joblib")
```

저장 후에는 **재로드한 객체의 예측이 같은지 확인**합니다. 저장은 쉽지만 확인을 건너뛰면 나중에 문제가 생깁니다.

> ➕ **심화 학습하기 — 탐색 기법은 더 있습니다**  
> 오늘 쓰는 `GridSearchCV`는 격자의 모든 칸을 다 시도합니다. 조합이 많아지면 비싸지므로 대안들이 있습니다 — **`RandomizedSearchCV`**(정해진 횟수만 무작위로 시도. 중요한 파라미터가 소수일 때 같은 예산으로 더 넓게 탐색), **`HalvingGridSearchCV`**(적은 자원으로 후보를 걸러내고 유망한 것에 집중), **베이지안 최적화**(이전 시도 결과로 다음 시도를 정함. `Optuna` 라이브러리가 대표적). 실무에서는 조합이 수백 개를 넘으면 Random이나 베이지안으로 갑니다. 이 도구들은 탐색 비용을 줄이지만, 평가 분리와 누수 점검은 별도로 설계해야 합니다.

> 📌 **실무 연결하기**  
> 실험 기록의 중요성은 규제 산업에서 특히 큽니다 — **금융** 신용모형은 감독기관에 모델 개발 이력을 제출해야 하고, **헬스케어** 진단 모델은 임상 검증 절차를 문서화해야 하며, **제조** 품질 모델은 라인 이관 시 재현 절차가 요구됩니다. 실험 기록은 문서 작업이 아니라 **자산**입니다.

## 📊 데이터로 확인해 봅시다 — 파이프 안에서 실험하기

먼저 `Pipeline`을 통째로 `GridSearchCV`에 넣습니다. 모델의 `C`와 **전처리의 대치 전략**을 함께 튜닝합니다.

> 🤔 **예상하기**  
> 로지스틱 회귀의 `C` 4개 × 대치 전략 2개 = 8조합을 5겹으로 돌립니다. 튜닝 후 점수가 기본값보다 얼마나 오를 것 같습니까? 0.01 정도 오를까요, 아니면 그보다 훨씬 작을까요?

▶️ **코드 실행하기 · 코드 셀 14 [C14]**

In [ ]:
# ─────────────────────────────────────────────
# [C14] Part 4 · 📊 데이터로 확인해 봅시다 — 파이프 안에서 실험하기
# Pipeline 통째로 GridSearchCV — 전처리 옵션까지 함께 (실행하면 best와 변화량이 나옵니다)
# ─────────────────────────────────────────────
pipe_tune = Pipeline([("pre", make_pre()),
                      ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

grid = {
    "clf__C": [0.01, 0.1, 1, 10],                        # 모델 파라미터
    "pre__num__impute__strategy": ["median", "mean"],    # 전처리 옵션까지 튜닝
}

gs = GridSearchCV(pipe_tune, grid, cv=cv, scoring="accuracy", n_jobs=-1)
gs.fit(X, y)

n_comb = len(grid["clf__C"]) * len(grid["pre__num__impute__strategy"])
print(f"탐색 공간: {n_comb}조합 × 5겹 = {n_comb * 5}회 학습")
print(f"파라미터 이름 규칙: 단계명__파라미터명 (중첩은 __로 이어 붙임)")
print()
print(f"best 조합: {gs.best_params_}")
print(f"best CV : {gs.best_score_:.4f}")

base = cross_val_score(pipe_tune, X, y, cv=cv, scoring="accuracy")
print()
print(f"기본값 CV {base.mean():.4f} → 튜닝 후 {gs.best_score_:.4f}   변화 {gs.best_score_ - base.mean():+.4f}")

탐색 공간: 8조합 × 5겹 = 40회 학습
파라미터 이름 규칙: 단계명__파라미터명 (중첩은 __로 이어 붙임)

best 조합: {'clf__C': 10, 'pre__num__impute__strategy': 'mean'}
best CV : 0.8227

기본값 CV 0.8204 → 튜닝 후 0.8227   변화 +0.0022

> 🎯 **[C14] 확인하기**  
> 변화가 **+0.0022**입니다. 40회를 학습해서 0.2%p를 얻었습니다.

`best` 조합은 `{"clf__C": 10, "pre__num__impute__strategy": "mean"}` — 둘 다 기본값이 **아닙니다**(기본은 `C=1`·`median`). 즉 튜닝이 다른 설정을 찾아냈고, 그 결과가 **0.8204 → 0.8227**입니다.

문제는 그 이득의 크기입니다. **0.0022는 이 5겹 CV에서 관찰된 겹별 변동보다 작습니다.** 겹별 표준편차와 평균 차이를 직접 비교해 통계적 유의성을 판정할 수는 없지만, 이 결과만으로 실질적 개선을 주장하기에는 근거가 부족합니다. 실험 기록표에는 이렇게 적습니다 — _"로지스틱 `C`·대치 전략 8조합 탐색. best는 기본값과 달랐으나 이득 +0.002로 표준편차 미만. 채택 여부 무의미."_ 이 기록이 있으면 다음 사람이 같은 40회를 다시 돌리지 않습니다.

파라미터 이름 규칙도 확인합니다. **`pre__num__impute__strategy`** — 전처리 옵션이 튜닝 대상이 됐습니다. 전처리를 `Pipeline` 밖에서 했다면 이 실험은 애초에 불가능했습니다.

그럼 무엇을 당겨야 성능이 오를까요? 세 레버를 나란히 재봅니다.

▶️ **코드 실행하기 · 코드 셀 15 [C15]**

In [ ]:
# ─────────────────────────────────────────────
# [C15] Part 4 · 📊 데이터로 확인해 봅시다 — 파이프 안에서 실험하기
# 레버별 효과 비교 — 피처 vs 모델 vs 튜닝 (실행하면 각 레버의 순증이 나옵니다)
# ─────────────────────────────────────────────
def cv_of(num_cols, cat_cols, clf):
    """주어진 피처 구성과 모델의 계층 5겹 CV 정확도를 돌려줍니다."""
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), num_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])
    return cross_val_score(Pipeline([("pre", pre), ("clf", clf)]),
                           df[num_cols + cat_cols], y, cv=cv, scoring="accuracy").mean()


LR = lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
RAW = ["pclass", "age", "sibsp", "parch", "fare"]
DERIVED = RAW + ["family_size", "is_alone", "fare_per_person"]

s1 = cv_of(RAW, ["embarked"], LR())                          # 최소 구성
s2 = cv_of(RAW, ["embarked", "sex"], LR())                   # 강한 원본 피처 추가
s3 = cv_of(DERIVED, ["embarked", "sex"], LR())               # 파생 3개 추가
s4 = cv_of(DERIVED, ["embarked", "sex", "who"], LR())        # 도메인 피처 추가
s5 = cv_of(DERIVED, ["embarked", "sex", "who"],
           RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))  # 모델 교체

gs_rf = GridSearchCV(
    Pipeline([("pre", make_pre()), ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
    {"clf__n_estimators": [200, 400], "clf__min_samples_leaf": [1, 3, 5],
     "clf__max_depth": [None, 8]},
    cv=cv, scoring="accuracy", n_jobs=-1).fit(X, y)
s6 = gs_rf.best_score_                                        # 튜닝

steps = [("① 수치 5개 + embarked", s1, None),
         ("② + sex", s2, s2 - s1),
         ("③ + 파생 3개", s3, s3 - s2),
         ("④ + who", s4, s4 - s3),
         ("⑤ 모델 교체 → RandomForest", s5, s5 - s4),
         ("⑥ 튜닝 (RF 12조합)", s6, s6 - s5)]
for label, val, delta in steps:
    d = f"   ({delta:+.4f})" if delta is not None else ""
    print(f"  {label:30} {val:.4f}{d}")

print()
print(f"  피처 레버 ①→④   {s4 - s1:+.4f}   ← 데이터·피처")
print(f"  모델 레버 ④→⑤   {s5 - s4:+.4f}")
print(f"  튜닝 레버 ⑤→⑥   {s6 - s5:+.4f}")
print(f"\n  RF best: {gs_rf.best_params_}")

  ① 수치 5개 + embarked             0.7026
  ② + sex                        0.7935   (+0.0909)
  ③ + 파생 3개                      0.8014   (+0.0079)
  ④ + who                        0.8204   (+0.0191)
  ⑤ 모델 교체 → RandomForest         0.8170   (-0.0034)
  ⑥ 튜닝 (RF 12조합)                 0.8395   (+0.0225)

  피처 레버 ①→④   +0.1178   ← 데이터·피처
  모델 레버 ④→⑤   -0.0034
  튜닝 레버 ⑤→⑥   +0.0225

  RF best: {'clf__max_depth': 8, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 400}

> 👀 **[C15] 결과읽기**  
> 이 표는 이 데이터와 고정된 5겹 분할에서 관찰한 결과입니다.

| 단계                       | CV 정확도 | 순증        |
| -------------------------- | --------- | ----------- |
| ① 수치 5개 + `embarked`    | 0.7026    | —           |
| ② + `sex`                  | 0.7935    | **+0.0909** |
| ③ + 파생 3개               | 0.8014    | +0.0079     |
| ④ + `who`                  | 0.8204    | +0.0191     |
| ⑤ 모델 교체 → RandomForest | 0.8170    | **−0.0034** |
| ⑥ 튜닝 (RF 12조합)         | 0.8395    | +0.0225     |

레버별로 묶으면 이렇습니다.

| 레버                    | 효과        |
| ----------------------- | ----------- |
| **① 데이터·피처** (①→④) | **+0.1178** |
| ② 모델 (④→⑤)            | **−0.0034** |
| ③ 튜닝 (⑤→⑥)            | +0.0225     |

이 실행에서는 피처 레버의 변화량이 +0.1178로 가장 큽니다. `sex` 하나를 추가했을 때 +0.0909가 관찰됐습니다. 이 비율을 다른 데이터와 모델에 일반화할 수는 없습니다.

더 눈여겨볼 것은 **모델 레버가 마이너스**라는 점입니다. 로지스틱 회귀를 RandomForest로 바꾸자 성능이 **떨어졌습니다**(−0.0034). 복잡한 모델로 교체하는 것만으로 성능이 좋아진다고 가정할 수 없습니다.

> 💡 **핵심짚기**  
> _"데이터·피처가 모델·튜닝보다 먼저다."_ 이 데이터에서는 방금 측정한 숫자가 이 순서를 뒷받침합니다. 성능이 안 나올 때 모델을 바꾸거나 튜닝을 돌리기 전에, **입력에 무엇이 빠졌는지** 먼저 물어야 합니다.

> ⚠️ **주의하기**  
> 그렇다고 튜닝이 무의미하다는 뜻은 아닙니다. 튜닝은 +0.0225를 실제로 만들어냈고, 그것도 값어치입니다. 요점은 **순서**입니다 — 피처가 부실한 상태에서 튜닝한 값은 나중에 피처를 고치면 무효가 되어 다시 튜닝해야 합니다. 피처 구성이 바뀌면 적절한 하이퍼파라미터도 달라질 수 있으므로 피처 검토 뒤에 튜닝을 수행합니다.

> ⚠️ **주의하기 — 선택 점수와 최종 평가**  
> [C15]의 점수는 같은 데이터에서 후보를 비교한 내부 CV 결과입니다. 후보 선택까지 포함한 일반화 성능은 별도 테스트셋이나 바깥쪽 교차 검증(Nested CV)으로 평가합니다.

이제 이 실험들을 **기록으로** 남깁니다.

▶️ **코드 실행하기 · 코드 셀 16 [C16]**

In [ ]:
# ─────────────────────────────────────────────
# [C16] Part 4 · 📊 데이터로 확인해 봅시다 — 파이프 안에서 실험하기
# 실험 기록표 — 조건 → 지표 → 결정 (실행하면 오늘의 실험 이력이 표로 나옵니다)
# ─────────────────────────────────────────────
experiments = pd.DataFrame([
    {"조건": "수치 5 + embarked (최소)", "CV 정확도": s1,
     "결정": "기준선으로 채택"},
    {"조건": "+ sex", "CV 정확도": s2,
     "결정": "★채택 — 단일 피처 최대 효과 (+0.091)"},
    {"조건": "+ 파생 3개(family_size·is_alone·fare_per_person)", "CV 정확도": s3,
     "결정": "채택 — 효과는 작지만(+0.008) 해석에 도움"},
    {"조건": "+ who", "CV 정확도": s4,
     "결정": "★채택 (+0.019)"},
    {"조건": "모델 교체 → RandomForest", "CV 정확도": s5,
     "결정": "기각 — 오히려 하락(-0.003). 단, 튜닝 여지 확인 위해 ⑥ 진행"},
    {"조건": "RandomForest 튜닝 12조합", "CV 정확도": s6,
     "결정": "★최종 채택 (+0.023)"},
    {"조건": "로지스틱 C·대치전략 8조합 튜닝", "CV 정확도": gs.best_score_,
     "결정": "기각 — 이득 +0.002로 표준편차 미만"},
])
print("실험 기록표 — 조건 → 지표 → 결정")
print()
print(experiments.round(4).to_string(index=False))
print()
print("→ '결정' 열이 핵심입니다. 기각한 실험도 지우지 않고 남깁니다.")
print("  다음 사람이 같은 실패를 반복하지 않게 하는 것이 기록의 목적입니다.")

실험 기록표 — 조건 → 지표 → 결정

                                           조건  CV 정확도                                       결정
                         수치 5 + embarked (최소)  0.7026                                 기준선으로 채택
                                        + sex  0.7935               ★채택 — 단일 피처 최대 효과 (+0.091)
+ 파생 3개(family_size·is_alone·fare_per_person)  0.8014              채택 — 효과는 작지만(+0.008) 해석에 도움
                                        + who  0.8204                             ★채택 (+0.019)
                         모델 교체 → RandomForest  0.8170 기각 — 오히려 하락(-0.003). 단, 튜닝 여지 확인 위해 ⑥ 진행
                         RandomForest 튜닝 12조합  0.8395                          ★최종 채택 (+0.023)
                           로지스틱 C·대치전략 8조합 튜닝  0.8227                  기각 — 이득 +0.002로 표준편차 미만

→ '결정' 열이 핵심입니다. 기각한 실험도 지우지 않고 남깁니다.
  다음 사람이 같은 실패를 반복하지 않게 하는 것이 기록의 목적입니다.

> 👀 **[C16] 결과읽기**  
> 일곱 줄 중 **두 줄이 기각**입니다. 그리고 그 두 줄이 가장 값어치 있는 기록입니다.

- _"모델 교체 → 기각. 오히려 하락"_ — 다음 사람이 "랜덤포레스트 써보면 어때요?"라고 할 때 답할 수 있습니다
- _"로지스틱 튜닝 → 기각. 선택된 설정은 달랐지만 개선 근거가 부족"_ — 40회 학습을 다시 돌리지 않게 합니다

**기각한 실험을 지우는 것은 정보를 버리는 일입니다.** 성공한 조건만 남긴 기록표는 "이 결론에 어떻게 도달했는지"를 설명하지 못합니다.

마지막으로 최종 모델을 저장하고 재현되는지 확인합니다.

> ⚠️ **주의하기 — 저장 파일을 다시 열 때**  
> 두 가지를 확인합니다. **① 신뢰할 수 없는 `.joblib` 파일은 열지 않습니다** — 파일을 읽는 것만으로 그 안에 담긴 코드가 실행될 수 있습니다. **② 저장한 환경과 여는 환경의 라이브러리 버전을 맞춥니다** — `scikit-learn` 버전이 다르면 경고가 뜨거나 로드에 실패할 수 있으므로, 모델 파일을 남길 때 버전도 함께 기록합니다.

▶️ **코드 실행하기 · 코드 셀 17 [C17]**

In [ ]:
# ─────────────────────────────────────────────
# [C17] Part 4 · 📊 데이터로 확인해 봅시다 — 파이프 안에서 실험하기
# joblib 저장 → 재로드 → 예측 일치 확인 (실행하면 일치 여부와 파일 크기가 나옵니다)
# ─────────────────────────────────────────────
import os

best_model = gs_rf.best_estimator_
MODEL_PATH = "titanic_pipeline.joblib"

joblib.dump(best_model, MODEL_PATH)
loaded = joblib.load(MODEL_PATH)

pred_before = best_model.predict(X)
pred_after = loaded.predict(X)
same = np.array_equal(pred_before, pred_after)

print(f"저장 파일: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024 / 1024:.2f} MB)")
print(f"재로드 후 예측이 원본과 동일: {same}")
print()
print("저장된 Pipeline이 품고 있는 것")
inner_pre = loaded.named_steps["pre"]
print(f"  1. 전처리 — 결측 대치 기준값, 표준화 기준, One-Hot 범주 목록")
print(f"     변환 후 피처 {len(inner_pre.get_feature_names_out())}개")
print(f"  2. 모델   — RandomForest {loaded.named_steps['clf'].n_estimators}그루")
print()
print("→ 전처리 기준까지 함께 저장되므로 새 데이터에 '학습 때와 똑같은 변환'이 적용됩니다.")
print("  전처리 코드를 따로 관리하면 이 일관성이 언젠가 반드시 깨집니다.")

저장 파일: titanic_pipeline.joblib (2.76 MB)
재로드 후 예측이 원본과 동일: True

저장된 Pipeline이 품고 있는 것
  1. 전처리 — 결측 대치 기준값, 표준화 기준, One-Hot 범주 목록
     변환 후 피처 16개
  2. 모델   — RandomForest 400그루

→ 전처리 기준까지 함께 저장되므로 새 데이터에 '학습 때와 똑같은 변환'이 적용됩니다.
  전처리 코드를 따로 관리하면 이 일관성이 언젠가 반드시 깨집니다.

> 👀 **[C17] 결과읽기**  
> 재로드 후 예측이 저장 전 결과와 **동일**합니다. 그리고 파일 안에는 모델만 들어 있는 것이 아닙니다 — **전처리의 기준값**(대치용 중앙값, 표준화의 평균·표준편차, One-Hot이 아는 범주 목록)이 함께 저장됩니다.

이것이 `Pipeline`의 마지막 값어치입니다. 3개월 뒤 새 승객 데이터가 왔을 때, **학습 때와 똑같은 기준으로** 변환됩니다. 모델만 저장하면 전처리 기준과 범주 목록을 별도로 동일하게 관리해야 하며, 설정이 어긋나면 예측 입력 표현이 달라질 수 있습니다.

`Pipeline`의 값어치를 정리하면 넷입니다 — **① 전처리 누수 예방 ② 코드 단순화 ③ 전처리까지 튜닝 대상화 ④ 배포 시점의 일관성.**

## ⌨️ 백문이 불여일타! (4)

```
[문제]
실험을 하나 더 설계해 기록표에 추가합니다.

1) [C15]의 cv_of 함수를 써서, 파생변수를 하나씩만 넣었을 때의 효과를 각각 측정합니다.
   (family_size만 / is_alone만 / fare_per_person만)
   기준은 ② 구성(RAW + embarked + sex)입니다.
2) 세 파생변수 중 어느 것이 단독으로 가장 효과가 컸습니까?
3) 세 효과를 각각 더한 값과 [C15]의 ③ 순증(+0.0079)을 비교합니다.
   같습니까? 다르다면 왜 그럴까요?
```

▶️ **코드 실행하기 · 코드 셀 18 [C18]**

In [ ]:
# [C18] Part 4 · ⌨️ 백문이 불여일타! (4)
# ⌨️ 백문이 불여일타! (4) — 파생변수를 하나씩 넣어 개별 효과 재기

derived_cols = ["family_size", "is_alone", "fare_per_person"]

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- `cv_of(RAW + [col], ["embarked", "sex"], LR())`로 한 개씩 추가한 구성을 잴 수 있습니다.
- 기준값은 `s2`(② 구성)입니다. 각 결과에서 `s2`를 빼면 순증입니다.
- 3번은 "피처들이 서로 정보를 공유하면 효과가 단순히 더해지지 않는다"는 점을 생각합니다.

</details>

## 🚦 짚고 넘어가기

1. `pre__num__impute__strategy`라는 파라미터 이름이 무엇을 가리키는지 읽을 수 있습니까?
2. 세 레버(피처·모델·튜닝)의 효과를 오늘 수치로 말할 수 있습니까?
3. 실험 기록표에서 '기각' 행을 남겨야 하는 이유는 무엇입니까?

> ⏭️ **다음 학습 예고**  
> 개념 여정은 여기까지입니다. 이제 오늘의 퀴즈로 배운 것을 점검하고, **실습 노트북**에서 오늘의 도구를 실제 비즈니스 데이터에 적용해 **모델 카드 v5**를 완성합니다.

# ❓ 오늘의 퀴즈

배운 내용을 잠깐 확인해보겠습니다. 틀려도 괜찮습니다.

## 개념 퀴즈

1. 동료가 "CV 정확도 0.95를 달성했습니다. 표준편차는 0.003입니다"라고 보고했습니다. 무엇을 의심하고, 무엇을 물어야 합니까?
2. 오늘 전처리 누수의 부풀림이 0.0000이었습니다. 그래도 `Pipeline`을 써야 하는 이유를 세 가지 적습니다.
3. `Pipeline`으로 막을 수 없는 누수 유형은 무엇이고, 그것은 어떻게 막습니까?
4. 성능이 목표에 못 미칠 때, 오늘 배운 순서대로 무엇부터 시도해야 합니까?

## 코드 퀴즈

```
[문제]
빈칸(____)을 채워, Pipeline을 통째로 GridSearchCV에 넣어
DecisionTree의 max_depth와 전처리 대치 전략을 함께 튜닝합니다.
```

▶️ **코드 실행하기 · 코드 셀 19 [C19]**

In [ ]:
# [C19] ❓ 오늘의 퀴즈 · 코드 퀴즈
# 코드 퀴즈 — 빈칸(____)을 채워 Pipeline 통째 튜닝

quiz_pipe = Pipeline([("pre", make_pre()),
                      ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))])
# quiz_grid = {
#     "____": [3, 4, 5, 6],                       # clf 단계의 max_depth
#     "pre__num__impute______": ["median", "mean"], # 대치 전략
# }
# quiz_gs = ____(quiz_pipe, quiz_grid, cv=cv, scoring="accuracy").fit(X, y)
# print("best:", quiz_gs.____, "→ CV", round(quiz_gs.____, 4))

# 여기에 코드를 작성하세요

# 🎓 오늘의 정리

## 개념 연결 한눈에

```text
[Part 1] 피처를 만든다        파생·인코딩·스케일링 (pandas → sklearn 문법)
   │  그런데 전처리는 기준값을 '학습'한다
[Part 2] 데이터 누수          네 유형 · 검증 누수 재현(+0.054)
   │  조심하는 것으로는 못 막는다
[Part 3] Pipeline            학습되는 전처리의 fit 범위를 각 학습 조각으로 제한
   │  타깃 누수 +0.060 vs 전처리 누수 +0.000 — 누수 유형에 따라 영향이 다르다
[Part 4] 누수 없는 실험       Pipeline 통째 튜닝 · 기록표 · 모델 저장
   ▼
피처 +0.118 / 모델 −0.003 / 튜닝 +0.023 — 이 데이터의 다음 실험 우선순위
   ▼ 다음 시간으로
조립은 끝났다. 그런데 정답이 없는 데이터라면?
```

## 한 장 정리

| 개념                | 한 줄 요약                                                   |
| ------------------- | ------------------------------------------------------------ |
| 파생변수            | 원본 조합으로 새 관점을 만듦. **가설이므로 검증 필요**       |
| 파생변수 점검       | 업무 가설 · 예측 시점 가용성 · 추가 효과 검증                |
| 인코딩              | 순서 없는 범주는 One-Hot. Ordinal은 **가짜 순서**를 만듦     |
| 스케일링            | 거리·계수 기반 모델에 필요, 트리에는 불필요                  |
| 데이터 누수         | 평가에 쓸 정보가 학습·선택에 미리 관여하는 사고              |
| 누수 유형 4종       | ① 전처리 ② 타깃 ③ 검증 ④ 시간·중복                           |
| 누수 점검 신호      | 예상 밖의 점수 · 기준 모델과의 큰 격차 · 개발-운영 성능 차이 |
| `Pipeline`          | 주어진 데이터에만 `fit` → 학습되는 전처리의 누수 위험을 낮춤 |
| `ColumnTransformer` | 열 그룹마다 다른 전처리를 배정해 이어 붙임                   |
| `단계명__파라미터`  | 중첩은 `__`로 이어 붙임. **전처리도 튜닝 대상**이 됨         |
| 실험 기록표         | 조건 → 지표 → **결정**. 기각 기록도 남김                     |
| `Pipeline` 저장     | 전체를 통째로 저장하면 전처리 기준값까지 함께 담김           |
| 레버 우선순위       | **데이터·피처 → 모델 → 튜닝** (오늘 수치로 확인)             |

## 진단 → 다음 행동 매핑

| 오늘 만난 신호                  | 자연스러운 다음 행동                          |
| ------------------------------- | --------------------------------------------- |
| CV 점수가 기대보다 훨씬 높다    | 누수 의심 → 피처의 출처와 `fit` 위치 점검     |
| CV 표준편차가 작다              | 안정성 정보로 해석하되 누수는 별도 감사       |
| 전처리를 `Pipeline` 밖에서 했다 | 위험 크기와 무관하게 안으로 옮김              |
| 튜닝 결과가 기본값이었다        | 기록에 남기고 **피처 레버로 이동**            |
| 성능이 목표에 못 미친다         | 피처 → 모델 → 튜닝 순으로 시도                |
| 모델을 저장했다                 | **재로드 후 예측 일치 확인** (건너뛰지 말 것) |

## 🙋 자주 묻는 질문 (FAQ)

**Q1. 전처리 누수가 0.0000이었는데, 그럼 `Pipeline`은 과잉 아닌가요?**  
A. 아닙니다. 세 가지 이유가 있습니다 — ① 이 데이터가 891행이라 통계량이 안정적이었을 뿐이고 ② 피처 선택·타깃 인코딩 같은 강한 변환이 섞이면 [C11]처럼 약 +0.060까지 벌어지며 ③ `Pipeline`은 전처리 순서와 기준값을 한 객체로 관리해 코드 중복과 운영 불일치도 줄입니다.

**Q2. 파생변수를 많이 만들면 좋은 건가요?**  
A. 아닙니다. 오늘 ⌨️(4)에서 확인했듯 파생변수 3개의 효과가 +0.0079뿐이었던 이유는 **서로 정보가 중복**됐기 때문입니다(`is_alone`은 `family_size`의 일부). 반면 `sex` 하나가 +0.0909였습니다. **"이 표현이 모델에 추가로 유용한가"** 를 같은 검증 조건에서 확인합니다.

**Q3. 튜닝은 언제 해야 하나요?**  
A. **피처 구성을 정한 뒤** 수행합니다. 이유는 효과 크기(+0.023)만이 아닙니다 — 피처를 나중에 고치면 앞서 찾은 최적 하이퍼파라미터가 **무효가 되어 다시 튜닝**해야 합니다. 피처를 확정한 뒤 튜닝하면 한 번으로 끝납니다.

**Q4. `RandomizedSearchCV`나 `Optuna`는 안 배워도 되나요?**  
A. 필요할 때 문서를 보고 쓰면 되는 도구입니다. 오늘 ➕ 박스에서 이름과 쓰임을 소개했고, `GridSearchCV`를 이해했다면 사용법은 거의 같습니다(`n_iter`로 시도 횟수를 정하는 정도의 차이). **중요한 것은 도구가 아니라 "모델 선택과 최종 평가를 분리했는가"** 이고, 그것은 도구를 바꿔도 사람의 몫으로 남습니다.

## 📖 핵심 용어 사전

| 용어                      | 한 줄 정의                                          |
| ------------------------- | --------------------------------------------------- |
| 파생변수(Derived Feature) | 기존 컬럼을 조합해 만든 새 피처                     |
| 인코딩(Encoding)          | 범주형 값을 숫자로 바꾸는 변환                      |
| One-Hot 인코딩            | 범주마다 0/1 열을 따로 만드는 방식                  |
| Ordinal 인코딩            | 범주에 정수를 배정. 순서 없는 범주엔 부적합         |
| 스케일링(Scaling)         | 수치형의 단위·범위를 맞추는 변환                    |
| 데이터 누수(Data Leakage) | 평가용 정보가 학습·선택에 미리 관여하는 사고        |
| 타깃 누수(Target Leakage) | 정답 정보나 예측 시점 이후 정보가 피처에 섞임       |
| 검증 누수                 | 테스트 점수를 보고 설정을 고르는 것                 |
| `Pipeline`                | 전처리와 모델을 순서대로 묶어 한 객체로 만드는 도구 |
| `ColumnTransformer`       | 열 그룹마다 다른 전처리를 배정하는 도구             |
| 실험 기록표               | 조건 → 지표 → 결정을 남기는 이력 표                 |
| `joblib`                  | 학습된 객체를 파일로 저장·재로드하는 도구           |

> ⏭️ **다음 학습 예고**  
> 오늘은 좋은 입력을 만들고, 학습되는 전처리를 `Pipeline`으로 묶고, 실험 기록과 모델을 저장하는 과정을 연결했습니다. 그런데 지금까지 우리가 다룬 모든 문제에는 공통점이 하나 있었습니다 — **정답(`survived`, `churned`)이 있었다**는 것입니다. 다음 시간에는 정답이 아예 없는 데이터를 만납니다. 아무도 "이 고객은 A그룹"이라고 알려주지 않을 때, 비슷한 것들을 어떻게 묶어낼까요? 그리고 그렇게 만든 그룹을 오늘 배운 `Pipeline`에 **새 피처로 다시 넣을** 수도 있습니다.

# 📝 오늘의 과제

## 🧪 실습 노트북

오늘의 도구를 실제 비즈니스 데이터에 적용해 **모델 카드 v5**를 완성하는 실습은 **실습 노트북**에서 진행합니다. 핵심 개념을 설명할 수 있으면 실습 노트북으로 넘어갑니다.

실습에서 하는 일은 이렇습니다.

1. 파생변수 2개를 설계하고 **가설을 먼저 적은 뒤** 효과를 검증
2. 수치·범주 분기 전처리를 `ColumnTransformer` + `Pipeline`으로 구성
3. **누수 버전과 교차 적합 인코더를 포함한 `Pipeline` 버전의 CV 점수를 비교**해 차이를 기록
4. `Pipeline` 통째로 `GridSearchCV` (전처리 옵션 1개 포함)
5. **실험 기록표**(조건 → 지표 → 결정)와 **모델 카드 v5** 완성, `joblib` 저장

## 📝 미니과제 — 누수 감사 보고서

동료가 만든 노트북을 코드리뷰한다고 가정하고, **누수 감사 체크리스트**를 작성해 제출합니다.

**제출물 — 다음 형식의 체크리스트와 그 적용 결과**

```markdown
## 누수 감사 체크리스트

- [ ] ① 전처리: 데이터에서 학습되는 전처리의 `fit`이 `Pipeline` 안에서 일어나는가?
- [ ] ② 타깃: 타깃으로 만든 피처가 있는가? 있다면 학습 행 자체의 타깃이 섞이지 않도록 교차 적합하는가?
- [ ] ② 타깃: 예측 시점에 존재하지 않는 컬럼이 섞이지 않았는가?
- [ ] ③ 검증: 하이퍼파라미터를 테스트셋 점수로 고르지 않았는가?
- [ ] ④ 시간·중복: 같은 개체가 학습·검증에 나뉘어 들어가지 않았는가?
- [ ] 신호: CV 점수가 기대보다 높거나 표준편차가 유난히 작지 않은가?

## 적용 결과 (본인 실습 노트북에 적용)

| 항목 | 판정      | 근거 |
| ---- | --------- | ---- |
| ...  | 통과/위험 | ...  |
```

**평가 기준**

| 축          | 기준                                                  |
| ----------- | ----------------------------------------------------- |
| 완전성      | 네 유형을 모두 점검하는 항목이 있는가                 |
| 실행 가능성 | 각 항목이 코드를 보고 판정할 수 있는 형태인가         |
| 자기 적용   | 본인 실습 노트북에 실제로 적용해 판정 근거를 적었는가 |

**제출:** 개인 공개 저장소 main에 커밋·푸시하고 저장소·커밋 링크를 제출합니다.

> 📌 **실무 연결하기**  
> 이 체크리스트는 앞으로 여러분이 코드리뷰할 때 실제로 쓰는 도구가 됩니다. 그리고 **AI가 준 전처리 코드를 검수할 때** 가장 먼저 돌려보는 것이 ① 항목입니다 — `fit`의 대상·피처 생성 시점·분할 단위를 확인하는 점검이 오늘 배운 것의 가장 실용적인 쓰임입니다.

오늘 여러분은 **입력을 만드는 손**과 **새는 곳을 찾는 눈**을 함께 얻었습니다. 그리고 하나를 더 배웠습니다 — 성능이 안 나올 때 모델을 바꾸거나 튜닝을 돌리기 전에 **입력에 무엇이 빠졌는지 먼저 묻는 것**입니다.

이 데이터에서는 `sex`를 추가했을 때의 변화량이 모델 교체와 튜닝의 변화량보다 컸습니다. 다만 각 변화량은 그 앞 단계까지의 구성에 따라 달라지므로, 다른 문제에서는 **레버의 우선순위를 미리 가정하지 않고** 같은 검증 조건에서 직접 비교합니다.

오늘도 한 걸음, 수고하셨습니다! 🎉

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>